# Kaggriculture Goose Portfolio | Historical LB 2615

**A complete agent, with a small economic decision worth reusing.**

Should a farm buy two cows, one cow and one goose, or two geese? Extra eggs earn money,
but they also lower the price received by the rest of the herd. This agent compares the
three choices before changing its purchase and placement plan.

**Run all cells to create `main.py` and `submission.tar.gz`.** The archive is ready to
select for a Kaggriculture submission. Running this notebook does not submit it for you.
The default settings recreate our S38 file byte for byte.

**About 2615.0:** this is the historical public LB score of S38, submitted on 29 August
2026 (ref `55859195`), as observed on 6 September. It is not a guaranteed score for a new
submission. The same code later received 1833.3 under a different submission.

The base farm controller comes from [The Moon Counts Melons — prvsiyan, V29](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-the-moon-counts-melons?scriptVersionId=345469411),
under Apache-2.0. Our additions handle livestock valuation, compatible purchases and
placement, market ordering, and the second-goose guard explained below.

## 1. Quick start

1. Leave the settings below unchanged for the historical agent, or edit them for an experiment.
2. Run all cells. The notebook checks the source, runs complete local games, and builds the archive.
3. Save a completed version. On the competition's **Submit Agent** page, select that
   notebook version and `submission.tar.gz`, or download and upload the archive yourself.

No GPU or external dataset is needed. Enable Internet for the notebook: the setup cell
installs `kaggle-environments==1.32.7` if needed. Kaggle's default image can contain an
older game version with different mechanics. The submitted agent itself uses only the
standard library and needs no internet.


In [1]:
# These settings affect the actual file that will be packaged.
SECOND_GOOSE_BUFFER = 2000.0  # Extra projected coins required from the second goose.
TOTAL_BUFFER = 600.0         # Required advantage over the two-cow default.
SMOKE_SEEDS = (38001, 38002)  # Each seed is tested in both player positions.


In [2]:
import importlib.metadata
import subprocess
import sys

ENGINE_PIN = "1.32.7"
if importlib.metadata.version("kaggle-environments") != ENGINE_PIN:
    setup = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--no-deps",
         "kaggle-environments==" + ENGINE_PIN],
        capture_output=True, text=True, timeout=180,
    )
    if setup.returncode:
        print((setup.stdout + setup.stderr)[-3000:])
        raise RuntimeError("Engine setup failed. Enable notebook Internet and run again.")
assert importlib.metadata.version("kaggle-environments") == ENGINE_PIN
print("Test engine:", ENGINE_PIN)


## 2. Why the second goose needs a buffer

The model estimates the remaining livestock revenue for all three portfolios, including
the existing herd. It subtracts the new animals' purchase cost and an allowance for feed
and worker time. This catches a common mistake: valuing extra production without counting
the lower price on production the farm already has.

| Choice | Historical rule |
|---|---|
| Consider geese | An egg-consuming shop is already visible |
| Move away from two cows | The alternative must beat them by at least 600 projected coins |
| Keep two geese | The second goose must add at least 2,000 over the mixed herd |

If two geese fail the second test, keep one goose only if the mixed herd still clears
the first test. Otherwise keep two cows. These buffers are heuristics, not universal
constants. The examples below use invented values so the boundary is easy to inspect.


In [3]:
import math

def choose_portfolio(scores, egg_demand_visible=True,
                     total_buffer=600.0, second_goose_buffer=2000.0):
    """Return 0, 1 or 2 geese from projected net values; this emits no game action."""
    if set(scores) != {0, 1, 2}:
        raise ValueError("Provide exactly the scores for 0, 1 and 2 geese")
    if not all(math.isfinite(float(v)) for v in scores.values()):
        raise ValueError("Scores must be finite")
    if not all(math.isfinite(v) and v >= 0 for v in (total_buffer, second_goose_buffer)):
        raise ValueError("Buffers must be finite and nonnegative")
    if not egg_demand_visible:
        return 0
    chosen = max((0, 1, 2), key=lambda g: scores[g])
    if scores[chosen] < scores[0] + total_buffer:
        chosen = 0
    if chosen == 2 and scores[2] < scores[1] + second_goose_buffer:
        chosen = 1 if scores[1] >= scores[0] + total_buffer else 0
    return chosen

# Invented projected values, not observations from competition replays.
examples = [
    ("No visible egg demand", {0: 10000, 1: 14000, 2: 18000}, False),
    ("Small total advantage", {0: 10000, 1: 10200, 2: 10599}, True),
    ("Keep a mixed herd", {0: 10000, 1: 13000, 2: 14000}, True),
    ("Second goose earns its buffer", {0: 10000, 1: 13000, 2: 15000}, True),
    ("Mixed herd also fails", {0: 10000, 1: 10500, 2: 12000}, True),
]
print(f"{'Invented scenario':38s} {'V0':>7} {'V1':>7} {'V2':>7} {'Geese':>6}")
for label, scores, visible in examples:
    result = choose_portfolio(scores, visible)
    print(f"{label:38s} {scores[0]:7.0f} {scores[1]:7.0f} {scores[2]:7.0f} {result:6d}")
assert [choose_portfolio(s, v) for _, s, v in examples] == [0, 0, 1, 2, 0]
assert choose_portfolio({0: 10000, 1: 10600, 2: 10600}) == 1
assert choose_portfolio({0: 10000, 1: 10000, 2: 10000}) == 0


### Where the decision changes

The axes are forecast advantages over two cows, in coins—not leaderboard gains.
This map assumes visible egg demand and space for two animals.


In [4]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np

advantages = np.linspace(-1000, 6000, 141)
decision = np.array([[choose_portfolio({0: 0.0, 1: x, 2: y})
                     for x in advantages] for y in advantages])
fig, ax = plt.subplots(figsize=(8.5, 5.5), layout="constrained")
colors = ListedColormap(["#e7edf4", "#36a6a0", "#d79b36"])
mesh = ax.pcolormesh(advantages, advantages, decision, cmap=colors,
                     norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3), shading="nearest")
ax.set(xlabel="One goose + one cow: advantage over two cows (projected coins)",
       ylabel="Two geese: advantage over two cows (projected coins)",
       title="S38 portfolio rule: when to keep one cow")
bar = fig.colorbar(mesh, ax=ax, ticks=[0, 1, 2], pad=0.03)
bar.ax.set_yticklabels(["2 cows", "1 cow + 1 goose", "2 geese"])
ax.spines[["top", "right"]].set_visible(False)
plt.show()


### Try a different market

The next two cells expose the historical valuation functions and an invented farm.
Changing the starting egg stock, with everything else fixed, moves the recommendation
from two geese to a mixed herd and then to two cows. These stock values were chosen to
show the transitions; they are not a sample of competition games.

The estimate assumes daily feeding and care, approximate rival production, and a
30-day season. Unknown future shops receive a discounted demand estimate (weight 0.35).
It does not simulate worker travel, failed purchases or shed overflow. Treat its output
as a comparison between choices, not an exact profit forecast.


In [5]:
# Reference functions from the S38 valuation layer; names preserved.
_E157_CONFIG = {'future_weight': 0.35}

def _get(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)

def _seat(obs):
    return 1 if int(_get(obs, "player", 0) or 0) == 1 else 0

def _farm(obs, seat):
    farms = list(_get(obs, "farms", []) or [])
    return farms[seat] if seat < len(farms) else {}

_SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}

_E155_PARAMS = {
    "WHEAT": (25, 400, "sqrt", .80, "log", .20),
    "CARROT": (35, 450, "hinge", 1., "sqrt", .70),
    "TOMATO": (60, 200, "hinge", .40, "sqrt", .60),
    "STRAWBERRY": (120, 100, "sqrt", .70, "linear", 1.60),
    "MELON": (250, 300, "log", .20, "sq", 3.60),
    "EGG": (50, 332, "hinge", .40, "log", .20),
    "MILK": (160, 122, "sqrt", .60, "linear", 1.60),
    "WOOL": (200, 105, "log", .20, "sq", 3.20),
    "FERTILIZER": (100, 200, "linear", .40, "linear", .40),
}

def _e155_shape(name, x, t):
    x = max(0., x)
    if name == "sq": return x * x
    if name == "sqrt": return math.sqrt(x)
    if name == "log": return math.log1p(x)
    if name == "log10": return math.log10(1. + x)
    if name == "hinge":
        u = x / t if t > 0 else x
        return u + 8. * max(0., u - 1.) ** 2 if t > 0 else x
    return x

def _e155_price(item, inventory, market=None):
    base, t, below, bt, above, at = _E155_PARAMS[item]
    p = _get(_get(market or {}, "params", {}) or {}, item, {}) or {}
    base, t = p.get("base", base), p.get("T", t)
    i0 = p.get("I0", 10000)
    if inventory < i0:
        shape, target, sign = p.get("below_func", below), p.get("below_target", bt), 1
    else:
        shape, target, sign = p.get("above_func", above), p.get("above_target", at), -1
    value = base + sign * target * base * _e155_shape(shape, abs(inventory - i0), t) / _e155_shape(shape, t, t)
    return max(1, int(round(value)))

_E157_ANIMALS = {"COW": ("MILK", 8, 2, 400), "SHEEP": ("WOOL", 6, 3, 500),
                 "GOOSE": ("EGG", 4, 1, 300)}

def _e157_herd(farm):
    return [dict(tile) for row in _get(farm, 'tiles', []) for tile in row
            if isinstance(tile, dict) and tile.get('animal') in _E157_ANIMALS]

def _e157_production(herd, day, end):
    production = {p: [0.] * end for p in ('MILK', 'WOOL', 'EGG')}
    for tile in herd:
        product, first, period, _ = _E157_ANIMALS[tile['animal']]
        placed = int(tile.get('placed_day', day))
        pending = float(tile.get('pending_care_bonus', 0))
        for future in range(day + 1, end):
            if future >= placed + first and (future - placed - first) % period == 0:
                production[product][future] += min(4. if tile['animal'] == 'GOOSE' else 6., 1. + pending)
                pending = 0.
            pending += 1.  # Production consumes OLD care before today's care.
    return production

def _e157_projection(obs, step, extra=None, quantity=0):
    """Finite-horizon own revenue INCLUDING cannibalization of the current herd.

    This is an approximate allocation model, not an exact engine simulator.
    Future shop identities and opponent private stock are never inspected.
    """
    day, end = step // 24, 30
    own = _e157_herd(_farm(obs, _seat(obs)))
    enemy = _e157_herd(_farm(obs, 1 - _seat(obs)))
    private = _get(obs, 'private', {}) or {}
    stored = [private.get('shed', {}), *(private.get('inventories', []) or [])]
    for animal in _E157_ANIMALS:
        own.extend({'animal': animal, 'placed_day': day + 1} for _ in
                   range(sum(int(v.get(animal, 0)) for v in stored)))
    additions = (extra if isinstance(extra, dict) else {extra: quantity}) if extra else {}
    for animal, count in additions.items():
        own.extend({'animal': animal, 'placed_day': day + 1} for _ in range(count))
    ours, theirs = _e157_production(own, day, end), _e157_production(enemy, day, end)
    market = _get(obs, 'market', {}) or {}
    inventory = _get(market, 'inventory', {}) or {}
    shops = list(_get(_get(obs, 'town', {}), 'unlocked_shops', []) or [])
    value = 0.
    for product in ('MILK', 'WOOL', 'EGG'):
        stock = float(inventory.get(product, 10000))
        daily = 1. + 6. * sum(2 if len(_SHOP_PRODUCTS.get(s, ())) == 1 else 1
                             for s in shops if product in _SHOP_PRODUCTS.get(s, ()))
        mean_shop = 6. * sum((2 if len(v) == 1 else 1) for v in _SHOP_PRODUCTS.values()
                             if product in v) / len(_SHOP_PRODUCTS)
        for future in range(day + 1, end):
            unseen = max(0, min(8, future // 3) - len(shops))
            demand = daily + _E157_CONFIG['future_weight'] * unseen * mean_shop
            q, rival = ours[product][future], theirs[product][future]
            # A public-production scenario, not a prediction of exact sell times.
            quote = _e155_price(product, stock + .5 * (q + rival) - .5 * demand, market)
            value += q * quote
            stock -= demand
            if quote > 1:
                stock += q + rival
    if additions:
        feed = _e155_price('WHEAT', inventory.get('WHEAT', 10000), market)
        # Existing labour route is preserved; service opportunity cost is charged.
        value -= sum(count * (_E157_ANIMALS[animal][3] + (end - day - 1) * (feed + 8.))
                     for animal, count in additions.items())
    return value


In [6]:
from copy import deepcopy

def hypothetical_farm():
    return {
        "player": 0,
        "farms": [
            {"tiles": [[{"animal": "COW", "placed_day": 1, "pending_care_bonus": 2},
                        {"animal": "COW", "placed_day": 1, "pending_care_bonus": 2}]]},
            {"tiles": [[{"animal": "COW", "placed_day": 2, "pending_care_bonus": 1}]]},
        ],
        "private": {"shed": {}, "inventories": []},
        "market": {"inventory": {"WHEAT": 10000, "MILK": 9900,
                                  "WOOL": 10000, "EGG": 10000}},
        "town": {"unlocked_shops": ["BAKERY", "BRUNCH_SPOT"]},
    }

def value_portfolios(observation, step=150):
    return {g: _e157_projection(observation, step, {"COW": 2-g, "GOOSE": g})
            for g in (0, 1, 2)}

print(f"{'Egg stock':>10} {'V1 - V0':>12} {'V2 - V0':>12} {'Suggested geese':>16}")
for egg_stock in (9400, 9500, 9600, 9700, 9800):
    obs = hypothetical_farm()
    obs["market"]["inventory"]["EGG"] = egg_stock
    scores = value_portfolios(obs)
    visible = any("EGG" in _SHOP_PRODUCTS.get(s, ()) for s in obs["town"]["unlocked_shops"])
    print(f"{egg_stock:10d} {scores[1]-scores[0]:12.1f} {scores[2]-scores[0]:12.1f} "
          f"{choose_portfolio(scores, visible):16d}")

# The model should read, not mutate, the caller's observation.
state = hypothetical_farm()
before = deepcopy(state)
assert value_portfolios(state) == value_portfolios(state)
assert state == before


## 3. Buying the animal is only the first step

The full agent makes this decision at step 150, where its existing schedule can support
two new animals. A first-shop Yarn Store follows the inherited sheep plan instead.

| Stage | What the controller must preserve |
|---|---|
| Buy | Enough cash when the order executes, within the ten-order limit |
| Build | A coop for each goose; a pasture for each cow |
| Pick up and place | The correct animal, worker and destination |
| Feed and harvest | Enough wheat and worker turns for the expected production |
| Sell | Actual shed stock, without displacing a necessary market order |

A mixed purchase needs an extra market order. In this particular schedule, one strawberry
seed order can move from step 150 to 151, before planting at 155. That timing is part of
the controller, not a general trick to copy into another route.

The implementation tracks intended replacements and checks observed inventory later.
It does not guarantee that every route succeeds against every opponent. End-of-game
selling covers reachable shed stock, not all carried goods or placed animals.

## 4. Full agent source

The following cells store the complete source and then write `main.py`. The inherited
Moon controller is kept together; our readable additions follow it. Keeping the original
bytes lets the default build check itself against the file that earned the historical score.

For a deeper change, edit the source strings below. `_e157_projection` estimates value;
`_e157_allocate` handles allocation and placement; the final portfolio wrapper limits
the number of geese. Editing the teaching examples alone does not change the agent.
The first two settings above do change the packaged file.

The inherited controller includes a fixed action schedule and repairs based on observations.
It is not a learned policy or a planner that rebuilds the entire farm route each turn.


### Inherited Moon controller


In [7]:
SOURCE_PARTS = []
SOURCE_PARTS.append(r'''"""moon_v189c_ramesh_yarn_sheep5: delayed schedule with lawful first-Yarn sheep cohort."""
import base64, copy, zlib

_BASE_NS={"__name__":"_v189_ramesh","__file__":"<embedded-ramesh>"}
exec(compile(zlib.decompress(base64.b85decode('c-pnQS(l<%)&TlmzamJ%22Fc83oVMGsNgK(RI5ZpKxC9b#p$<SG=_}G%&P8tzjfmw1J2oJ&*LHT+qZ9B%Q9akDYVc{{pZCIOf8Xm5iP^kB~N-835MiuUu=J^tKy4cZKM}LbS&3>$s}Kt$7b<mRLi|6zTj-!e}DV-?GM$kEysIV3$B#UzQ2i<9lSs9T+4iaKI`iGk3at4tF<xR@3AkX=kC9COW3&ZWAon|>A{R6xi0*Q^51JT_gLIW@V~^w{&wgui(6H=bWM%l(zsCVM(Sj7Ur*)+1=(q+ETL^!)jA~_r&bzSB#>djsh4JxbK6Rj(3#p_R{;;jw2Bxxx+fi3kn@=4weH2p<Q&ieD&6Y2^x4VJ$;61R=By<x0P$`9%5g>I0HAR6rV)&LOYlzTd^i>6ex@04ySrBRC{SB-i9~Ce?nVb%aaKpreFF)RmXT6pM4S>`rHm&81@x}%VzJ1daNb_6`hXe7HA}7pu~Fr+&+AybfEI4TnyKP?zF!;j_vu~YAPCChJ#!?}C4Y@<ePDK3BvQ2@oGe8kNQ)PiQL*HAuHAAo0MT*1u_`5caCCZW_2xluL~WYWX(AD;@KFHF4vOtr!rBG{XVkk4_4L+=FP76pFco5P#LjFlPFrsIl^Zo!70(P)&0|@470bYgjBb0qwtl$o+RWexS_4#oc9B_YAx_9$bXqc-T<6%zdHFFFGwmTGW>4C!X$x+Iaay*TUN8y1W%m+bYiqE@q5;LXc5xTGjqPK3p!;#^w%ZQOdo|j;to$Y^UlU>r97}X&*T4CjG{4h;{=At_@XJWs-xfNM-B2QOt9Myj`F5$>ZLr>Q5zWMA(<O^uPYoKHmGN~%+!d+?A8t#^2>{9}jl`IZ32amvbo*M1NYijG=i4(;=gP-jz{hv+k><`gF5;3=OOGH~K?m2<UIY)b*$@|&bXB`_h1lv66`7c4R)g~G+@T`e64*QK45pN*w&@gh!x7ionmU?==hqDZ2V*ZW?&0?3g7!Ox1~qaQ5|}De;<#S&odQ!UL*#sKq-Q{5HLYy9@wGph8!V_S<H))jEo4Zmw!0XK)SPRrlV!`8T!2+{GEX|!3LUdyx4<|)8T6|FSoU@}b8AXYqBjRuWM-2mI;I;P6!$xLKJM-Wh+}Zt990`eq*ag>?OKv)PXm9wZdK|la&J{u{Z99Y-?hegoS;2yOtVuR(vtN9DX8rIf^WCZ0nJm5TAi3VlVOcQlv^bj<<c`kiQU=?Cg%6!x;F&(E(x?^aF(4yWcq#?p;TfBf_0zLrtu^a<%=zD5IvfDzK*rq#<AT%!R|SlQZDelb(%$~!?0ahr@*4ScOvWJ7+G1BcH)Y}(oB6zlkEiqW~RBZo;Cg6bjAhlxR$B940*!Y!!*t}EHzqa$MY1=79&(M9!L=y-ki<}YM}*cJe>#P)`YO2mIG2#ROHiGv^nJD^h(UTf)%7KdYv_VU^&0Kxy`B3x=qn3a>bUX1#geI(orcH`htv+-P`~-lcO?L&m$0XMeAp(yvd*|@(e+JW{lyjLN%6c?#Sxa1=f*c58kZ0x%FzVRh33m>wvLN)JRUtO<+Z5YU7*BIe6AO1)vuxVI}?oqm69_OZh~tJG#@?!cqdF`($Ba;Kw9z8LYH<<&*`v(irx&#b!xj^1j8EHUk>4xK2=)wR_xbq%A~iS)gQaU^jBZx?XfE?T96Bv^Q4OPn3c@rPq^KdjjgaQUBx_r$pMG%2((_w&Ul)-q_46wH%-3h_udHi0k0sPHW#I>niKDQA=Bud7Pl#DL3||PT_j&E}K1}wW&>7d2BSh2%adeBmi4kyU9Lk7O3-nP!YvS-R5me-WwS$i&kq$*4Uh6uV?64x}@IHTt%D8E_PM|DFY{PY>~=!Hxvh)V$FN)JnYlho<a7|HDheUbFdAi?UvpMvUrn<h*7G3>}W_nLj-oYmRs!LTt+Ac>u#f41@F~na(Cbd!xEgY2&fZxC`3NA=4guFE{aOVU1%(ONo#96U2KWP_85o~sUu9N%-k{NZD!VLHuuwj?;P_ga}avR<mu8)PUW#eMp=K}(`q1hWz(rxx5a5!sJO@T&OTp+wz+$<nzbNzaOw>7QeO>j^=vN76}fCdN`gRd88qFU4kk0_rQ38mk&|lc&2`KsaGW9c$5^SwP<^Nu%I2l|HBmk-iLvVi{AwF7oQuVD=~lz0LY#r4zI7|8O7ZL`ugwr78Z^y~Pu&M&ISAERxs3{pXzACmYRnENFxk>s${l4d70wohmD>4|E;>7d=d9FN>Z?knm+9VbMS{pit4&$HHB^(haIuNFFsw-@1+vMs3>@>>TYm|kQkuFQCeU)TxM$Z(q0+&+jX?p56w(w@;Jo#<UB*+j)6!;q#Tjh7u-YQ4=OtVrsQ?zi0V^HIIGIjE7CtN7?IXUQQ%h_jM0rsbc4(5#+h@9iyOzBX(XHS&(v9hmCwJTY@Tyqd>C#zZb=wq@QISN)-C0&+t5K~nyd3UCD~X1Rt|SL}d!G=_<7i<dC7Q&HoHZ-E6}w&EV4^LT$Qjr3tH+Z=%k*vG<n~xt7LeN9Ee_qzDl%-fR!ywMxG8Lx0976^muvUXM#a$WCfP)N0^ZV%rb*NzM3i)5iJp?__e$t^(ok}Ib22*u$JSWnrX#B%YXb*$179UzgUVELR<i7FolS<LT6!N+lT|p2*)1k^&Fkq1bHb`@*Qf~b`0_kg=UwntvXZ1buxAh!HZfX5&YfelEJt={Erlu9nBJrSv~%?XNH+6}EzMNft22hg*-VDpW(}N`r+F(fTM?b}D4+*pf5mifMm?e3#Qvl@hGWqV&0T9Gku3;~`=spIrTC1^V4yZV)^A#>-i~)nXHuMTXM#k=Jpyv|ND&%W`@J-&#~PJ;<hnza(*jS>z?hao8zAD*ilu6Z*r~D;5#|gDE)On#I(E?~8ctkq2et3#wbGP6Y2cuKsuJzELOQigqJeidl3Y1-h#5wnFIY?<x^T;DA@x&YoJnJ|-Zl<qH5kMrn+DHd=|)9q?_##rGm|_xZ`%;G#AB9GT=~`@C%H4vP}^WhqtTvNx{bpACtwFAKI`|j2$goN0)j&+C0gtcNN8773wJ$Y%=TsHdXMaua$*=SG*5F@ixwiqQrJ4C($M0Xt5RUl=6up@12bD~V3U}48Nzrg1&TyE?|P!z&Q~VasVx_0iGqu*>-EDeJs;-lx7NTYjS;uE_Kj3=eocV}36wXNA)1YdD=|IhRuaVS<imPiTZ1C5#h7L*9|Jd$Y35G4iIg+9dyKPBp~x}=sx^7VCw-L9h71YSFuakpn*rE-mW)r685lEgZ0XmAw|Nl-w&Y+H>0MA#CBrg3qL=8bHe8xe>~;c}HdnMxrOX2Mz)sR>vCIs0Q^l-w->8eQ{0cH_>GjNoIF6g1kkAHVm}L%NHD5OKVoQbneLjmH{j|C-NhmUnj7fNs$NS_pGty{dT30(~Y~JnP6PR!i>WdRHO5O2EREOJI^4dTxj^Vj6w)eonnBN!hGjnAc%}7J(5i<#E1KPSay44rVG?85t7vpR#(mc{cBwe*|CP%K1x_q6M_L0GOfgaLlNvce!GbS%xC~Ti=<xX0O=Q}7_(ApR19<3bO+HTKOCZalqPx)P3M1-=~Ycw^{j-RwKdos7zL2SP7_4nx3T4p7=E63S%r4r`hTM0d}%%#;um!w@obN$ZR@2&fFvv*TAjbPYZ1t|!gkmI@LNdvWCZ!88&SV`Fyd6<-)BbsxT@$Ce@?&Iu;Tu<5LN^F~+mhClKKE(#*2$I4lXQH_E6C=EctJkc~?Dh&y)vtnc+1U+lBIF9SI%w!I|Abx245*a(wt8D_HzT*-Upw_NLObc8eQGOiQDdCB9BJ>&#xQ@G8v{v44p|{vXOsIxiIL*Bw9zi8P--7-?8Di|?o6^C?&ZSK>eSa(_C8o--BDxTFI3MXy^zXI<PKKp7o`*ffO3BgcjBF3Q85$EF<XY~s!^<)*NUDSO%7);$qrQXR?p9p@s(I@6SszoEpTempJV0tsZdHA!{~A`pceFKvAY!Ojmrwru;Y@gK)}|V76Nd88u+*M8KDI8fX5jzwHCJ(LIg7Lk(4w0c_o)jcGNK29~!g#cEV#el+f9(ke@ge9*k`xb0IywkkiZ{%OBf8#PU{|YGt=Sdnu|MMUZ<e)pfZsn#T|mA@$_G-(k*l#F#n>y52DQJ$-$Y2K;uEX0`~Xd1`T+<k2m;A2<?7wpxS9aJeXoLAuaT(-L*}!kW1VTlDxjawJo*q6p@+f@%BMG}`s_dSSKXz%A_TTWFqM`W)Xx=01JtH{!jKRLrYas|=s|`DulttcD@@#5w0pdG<sV4ksp}I>wbBE@Zi%GW&<ZPI5~5^EtgF`>|R+r|E%govz~HaL59gl*+USPNEJ?w&Zd0z7lQpS;#=K%fvX<WAq-Rr|ou8t;ELB+3A+(D_JfV+iUSfXBEs_%O-PO!i^OkWeO2LjN);3S=mmF4c!V>8^<b+X&fTd9?|yG{=Hq?z#N!fBlr3?-iM3Gh>(o&xV#$drdoPZO&vCRc82A-dFL>^Uk*b7MH+F$#hN*Yn4iVy5Z*qr-#)f%`71oe6UTeLy3A_zn_THtOVUJ{`5Exel&VV^lUQawuvuveuh40XgSxDT8hW+bz9yP!y>giKOY(6`&0}+Myhvl9U|wn%#fF>~xU0V>60p(U-?y?RUvJlzhwm@^K52v0ygaZrrB>t;@2^(A1Yrr}#>BSb>0aFvnL1Xbrrm+v<KWt2g~xXWaUf(|2KFM<E01r7qi$0)8#nGVT=)42%}k3TD_rUV`)I~=SofQ4Ux>r!B<g^Nwa+brtDr;Y2u?|X95K>Ct&m{2Q*C{%;uR2`rIpdCHmx7|aE#cT!vSiPoX2x#Yeq)0tJnlD?8Et=#F9#O08c8528Enck$o(A(9Ydl&?CbLAyyLszQs~hv)M-L;Ye&dohcRxAg^<xm$#G@iMi}?e7?*g=Sq&<PcC9zx6oM`+TTHz^vWl)=3#Zs$u1nsOI-T(bds80om9#QEj54(tu!syCPU3bPx0Z}-&OEvH@j@t#!Mz|!r*qXzf7H$mv!;jHBv{W2Yj2Fq8GO?5qA@*cqGfwjV`CLlw7i`aeTk0J1)3oI*a41oQwwrB1lVR9E{+EVPPOxYwPSL7Wahp@P49v#A(lIb)ie6%QjYtinG=FJQC!wH7ulBi#p%R_Iq7|vb>68$$lOsltinj(^J)VcfER`phe^gMekzm+G{6P^Pm<<b*LOHWE*WF4R!mKOt+SqBBn`b@Y!Gk`kaeuQ_YBEaAAULK7<7FHok5jxw5Z#SWE?x;SDav3b|U&tK5of7CS9HcA9W9t^1*x9bn>isVO#h#^SA9HM=H~Za4^a?h}m-0K)81ksDkfc@Dig5iWjYGaD`K0cG0kXGCe23R;DB>STm&B^A5tjIj1nmTzQ}PQKuum|W!+^y@dlWD(a)8RPVvJjP;F0SH=6B4F$*wTX1`OC(e6IDRTU98GU-)Xbby*139KR1lpri$e1jiz5BqX1%nEMOLEhz=96h@+uLuHb@Nv=^iR&&@din?(FOU?mVToE&(yB@&{4t%VeEu6Xl7O>Z}vz+9g{a7O%?UC@eFP8X4xd_Ng4_Q;m6-wA6u=J4HHP)p1z0G-_-&O;bT?=yF-^B@F5^nKw#NbJc6*d(id*R`6XpF#BO?=*^e0dsSYd=1k6xNA=1wbI45GUYf|qv)zM^L_KGaDZ)WQ;Jc+_?S3QfzNKSSs&S|f*t!+HO~#FSXR_~Y*1e0cnKid$v(QtFc9T7CX-Gd%mCK;p*1|fD<mQAE8_3=5Z9R{4_58xfPjg)_RbB*#Nj>kPXUbHEC3XfTn{vN3LI$_oTH&H%Ykc?4Zk(+MyUL&#jg4J8h^g2-#i^%Obz!%+L80SBt&ER=Ymh|Tb(sTqQ-?Ot$odKy{AzJdReFajjZUuG*%(-EuOZwovuG<H&X#pRzw)KqWEsP9x19!(SxKTG0J&5P7YA}T7qYoCs;ddjpT-uFn(Yu^>KCH!wTB79a>}gwL$kz@L0JczMFW(OiwLBYi!F7?%K9*;F4;s&mjTWaHmI{5x8egky;bISm%{dq&cK_L&fLs8aHF=()Z4f`=}7c`C`~SPxmr!rXs6ej5BI4P?#)?c66A4A9A7g$t<ekqTmjY%Z+Q0e>H-s8ga8|G3CYY>F;5u=<GHd@jBT&tBl||*$^$F%c!4#mwQfQ!4?%-ux7ZcRwztleO|x*$Y)4R**}_?-5-Tr?XArg1g~{ZyP205^C0Ozh%$4#H@DQW*&X_r~h4HM-rxSjBH)scOq!-lVqDHRw6(&8J_V<oVK%)tr>~)aak{OjFPWf2W!S!x0)`bPBbnseU4CeiHKbeDvW;a&36l#&uR9T4F_@Kb+wF>wGiK4Q<jNXmL4b^T7X_j$ko2!bMM6v3w<Owy9;dL()F&1L9mI}HGqHXC5<Q9}%D@>@F=D4^XPowatZ)E7|6$rd+A}?k0qZtoeth5Vky6H}uN+E?Vgr~BDTyX&9+wPgznEULtlLk8d@>uYf>X2ulH6-u5H*SZT?4prrVI3_{OBXvmk4=UAxY#=Tt0w7<h6{RbO!fMPQ3;&io%##D-z-aD6D{_!e0Ast#ilLP!06IBL@u~0RpSSJQ#`UqalGlZ&r73Hw<60cc+-W6;vHt)CWa6F;aRg+*;C){R7)*}1wGTPhuzE3D1-Vneo5dc(OB&4m1b1#CM>}jwX0hJRxtpZkN0AOcqZF*#h5Wfq0}b0=En))IHXDdd(TKdmTNIw_i~#hOE(Yg<qic?olQJV?g{~q8Fya~C!@(OpRbv^>pT5^N;kUkgFW+BQ#V)Z5XVGdkLJ@$g{tG{3m*Z~B@WHa3x&K}?u;1W1}v**Q>+zBNo~HN^W6flsZ|>Wmxjsx$p_&xwF}cvSp3j^{v1IeTk(ZTFoCDjOFPpKv)P{8U6);*i8a^*P`I`RVF(<0?e1==RYnV;RNl<P264aab8aMdXl$FQ{YhWub5UxX@R?IfXoU@=t%jMPy94Hx>vR-9z;oGVqx58upMY)vvowE65>UOCIBC8zn=r+6Ih8^>JujSVxl^7`Tor>G+d!0&AW#6IKq=KdJ%27WhXQMyH|s=DhMHb~vjA&>kD(hjd2RYzASkmX5*wpntaLsNuexh#3z;dfUUet7itBdMtxgWv$hk8iNCbtTR8ky7Y3f|@^OuX<j|}g`(L*3@wVZ9YDX9<ak_LNAFeP(YEp%&(Qv_I332WKtH;10$wkLqjaakDdGWDq<R;zG=Vwb=gSK?MjKE(IQJsS3RsTf?4OVwyK+r}=~G<$PAfT?X4kw*N~hayI-98ykJr}Spl$}6JcPp2!{vV(rQ=I@!ss-GbGJaJJ=jWRbRkJKm)Ku0~lY|gV*4D$h)TxO%P(B0bgMsx#VarrhXcnqKGrNi=|Z^EIhSe540BQa`ehE2fwc8)KVBG9233Xy7>?}M3A1(}#;*5uX$GZjv@fV?=~ImHl^$#sw$H1n5QU+s0usUuXdjpIoGvp(;s$6C5@h_Ux%R+}4MCe}Uc^ZHqwq-nTbs77*Id$>w5<uh91Xf%_7%I0Ruahu5`Cv`_^F7Lqr*&Y;WqdZbExUcf3+;Bb>q!O^>6ebx%3#ntdJ>Qki{2^@9c2uoizh7$+kZ`>7%I+<ScRDML9owXn@OD>94qL0i>XaSmD=C`Tuxz1{yj3FG9bi@k;y%Ob0Go5@-7)MN^F6mGjl50Dn@IpK)~Ch8#>j2Zk%Bo|l@=-};|6pjDjb(kCUr_q#MCu@ANH7RFQwroKkSbX7dT4&b_O|S4j5rOqyDhh<G?J`rx0Hg6rJee{RvQut1@#)=ypF6^>2LuU;(yJOG^$(ooZao_T5$r^(NEfunsX$&As9Sy#S>`<0<NTH%CsD`wcbd+kBtyr8#?1S~mvjKJFDa0?8aCkM{gyMmZ48J)Z2QkOmZAFeXhl#8cujTMk$0)h=Nl$_8CjQ$$X!M3eY!AW0|@l}lb+a_Bn&ZgOOzljSO%RDY8mYIvB;!PV8_6G)P)HILofK#qIEWN9%x?k@ICmS$l?bF0kC*+Dw8yGtC|5GL~~G&Iv^wO(kAp-8Yz4wFu&H<)I($%yOh!<v95@L0jF_26hRV<S8t_h6=jMfVZH2l+7Gn)y;}>hsj{SkOkpWNJQ+V}(GMc9KA`akLz4k|ek~;jQef>3S=^-`0p;72#JbO?#Vu1<a|<IS6X$)J5}lR_Blp#yAT1oMgtp-Sc*_iPT5xy^^a~>Fy!J?@N)BR%+EstQf(kH5{mIu4lg61x6#WHX2Z)q3ve|)KOm+D~*|aJ*4d-7QgcMt+IpBBGV{hUR%3Q)viL~Q+1gjh%>EnHlgbawAGITHV_df`Pt<ziaHq=<H>yP(5W|v$vu@(kP%;YPKtP+ouC*@khzVmZA*s;L9Y8qj9Zk1dPIh&kxsdFS=1KE<89pI25Sg+Auvex%*v9UmgKpbThxK85ZSKUMF*fZ*4Zo1M%Tl9yk2O#LNBqqi164=ohBF*HWESx88+J6y1Hwhj@+sj&w@RSMoP%Ad)#jJh$JSnjcGxfE%dlH^&3o`lmNQ}Mfu(W!?#4i$}59iI=f%txf9ch5Za88=1qNDK+{z+)w~xLbuu*q9IG<&=bV)46u3<<uC6s0V#v1JJT+SNW53#%*=j;<i<QHXy7^g^Hb?2f#F&R_&7TIl)O~56=ZBRDIVbeGFWb3(3aHlf3}Flp-atG+R-xabVkLb|IU*;o5k8?M-4b(5x`)LoGYal*OAUJE?y!$0x=dro1ngpc6bE^nWB`q%6zSr~OjNt21@*eg%+T9aJsz<57B}tQaFmkd127B<X&jk>#x<LovtbZq6|0=A4$DnnU0fTRe70N232UH`USto)S!=V^t=q1{qZ!~{Z+3Ik(B4N=AW|&*F~1ZIDpe6B6Y{)`y7BQ&gUa(gQg0xY<-Ht_o=iz;4)630H+IPsFw>DGRNzvRwyZl^4p)u#pnh5dB@|J>EblMSOmY)u3+R|fqKz^k!wBD%+|m_jifgt!>~=WwT3>B5Yi4pB<VMZYn9`Sp7R0Pp5=+&3<%5(?FSgdSY(ij`$>owexi!)p%DgvoqLLhLH8(W|)7W8J(le`TR!N^?SE*D!(dZ#r(38EHtXm9<XLyD~bjg9ECK{_WWFnY2g>`RQo;jssgd*;ElTs@{XWx<d0!A#{vdwP0-7>#Ov%Nmh<ihj@b;ZV}h0hOw$725AV2@&#m`#{U*ov*r{Zr8`YFT5SUj+6czUX9o*I1;yXw|1qJhRCpOK2-jSdC7c)A9bOOlBJO-MYt&MzNl~otK4}ebVJft8Qz2DynCXqtbQEokpnv8a-uN>p(GPjaIoaxHb!(j+ez#K0)7HXVeSwggb6$=K@X!dQ3J?L@9Bw(Q4yJ3>0*NpUfCkSQ%x_&RlyrDY)o#H)}drmeY*deQqN;pkA}@qydK$LzcyWB3=gy+i2J?PRr?LB#^PRpJ*==T90Lve3R?o3G%#a9;|+MoR1D?%F&uldTHF;9OjooSFS>67nq7bL#^RSbeP#v*M85_hJ=vqM=qsUBHX_M<i@Yu2MHM}R%?sZrgiDdLSoUW2+qc;cJc`<GvfJfW~E?LjiA_(yYPVAqRXEUY}j)P41`#TOwp!;txb&%C9s&V#z1|ENc8Zy=LWW(s!rB)(HHL*Zj|wM7c_6x8pXCbaw|u0s6b>m6tt@C?Q)eGoJ??-)dgV~Me<{#NY<{$XcAa-SNP3L?0s??0Tr~+FJqhYxLCc-l*_(jEti+-7$gc&#>)0&_0kzhSNc*rZIP@$j@IHzA*tu(s@ZC!{G2CQLvVKijZBcr#pZEENasjmhB$BqX=JX)GfJnjj$f+5a-P~8(O&A3IiWU`>r#nXu7Fi{l@YpKof=+D2BY`!RB%p{LQ4`|JTU7%HxWQ_k3bN7T<jqxw~aM-OJB*)mtCt`NH=yev$DHeD6Pb7iN9^9k(e-DW#n-hnPBB{YAJy3>Xzv1i%Cxfr(GptM6R{g1!$R_Wa|hU>tc7>Nv<&)zDyJ007$oQg0PCm3&*5)^?*6P%1-thg`U@xQjsqJgyu~!+pK1`&E1ULF93%KF-@<2K-ZlLkn4pEdo&UkRdI8WKxLBS>-|+a=pVv(9u9&k*~>)pwd}|MSR&~yvAR(bb{mB5B9}$qHj9PYI-X~}@+88Z`P*Uut!Lc%<{A0RIPZaxI?zNZptx6NrJR!_2S$@jR69#@Nn}cN99k5w+O{ZZT;p``dt5DNN?`knkDNwJTUxn=)wpG>WM<GUvf0ZW7@(%RuY$-~YL%#TVH3+v`vMiQZ}Vf8FWnkL4o4NORT_GCMWE}=$*QxMPVcMg{^;&deoHc^`XN%AyBpA`tYb;c<GO>^vUX)_9US(|)+|0t)|&HNA%ZH>V5c;$o%Ps1V0*UOyD-h!^e8m6bY#BmQ+;Dw0M5+1oG0TDOvPd>lb|+P<e-I)xHy6=*J0G&ag4XKX0Fg0Ht+EsaJeOG=|*u=H=$%_x33d+sfC(S)(-A*xPXb#`jW3@i0IYB>ycA1zcxDKSq<LSh4{3<rzTBymh$qkCYkG0s*pJ9<jJY)4{<M1g|B_^luH@Kq#jkVtZc^zKC6}q6xFv8P&d0NKrT=XY*XtsfM^@fX^9D<Oq)Cu<)+PbwOAV#ms@##G%rLo4P~;NPMyb=E*I(SCQ%IHbM4mhJ!+$v7J7gu8MZefa~Zia5(Z{BR}GNpcpF{y2LoqEqI5R`HPcOr8jYM@OLe?<1x=<R2!p8esfl?7wK)n07+!?-O*d1nOTr+<^n7Hyht#dJ0rI=ibdeRoY4kKKE#t~`9yc(Sa3`m97cM6D$4H#k>ZUY{!;MS51kCo{F*(Zz`Po3tRKPZtA4R6uxf@g`>8vf!DyZ3+Eco`Mdr@i`UCZYQp#n>dP2DP>*F!WnJhJ&4k-HGaYC;ziJRzP-R5Hnt_x+Y#_1Q#Zb&n~&-$M_Xv<DE`dc<mAUdL*(^OBaSndcZaEuCU0d$WCMp%F7tXdjL{Wwbg^=}u`rSRj)lbFwVwm>*l?vg02i=!y+%^?n_$0$a7tZsxup*`K0Tq6X^$(^-ne1q5IVEl>b<V6HAqBQb65!bI4dO-u&e(^ila5;=@nmG<;4AE%9>cdW=rHDL|aRf#<VlgRx<5`fc=u2v}G)bf;ZyIy3Z+`=(+GZyY;r3hcgcqYKmXbjwLnPN3@I<MO5af9~T7o<M-E5!iYQpbYK%jKT4pRIPF;|3CGi+FD{8r5u@9-76<ydOp&k-c}yzPmL?47e$Fh;buPAIRqcmWec1eZ}@~(9%|Rkvh_^<MY%t)@rMAzq~mto$`iR?1x!+lkZrRjBi%(@(`VCRU((F9NG|PUAj>lJ#L4U*f1VnGW_jORxb5;KGPr~8!wwKL%a2O7Bk5Ch%zH*MD&eVdU2`=$<gDRL?Qzc$Amk$clH23T+)SYazE4<Q7g3e7G`Wsj2%JuZWcfeLM2yEpCiE>9Y(7c0djMT<czM(^YfG4-!drV92^hJbXaa@LQ>p3F7hD2)@&y0BUCOy#r*<MXo*mqTejOxYLGu)=36s5!!W&Zx2=6~zBc9Dg~{2xHXd(i8}8i5WA1Q)lZck)^2)K21iC3E8pXusAy;eY>{t&*90x&yEUuN<6@uiGlI$MW4uiXi*k;bDrK>p|O|Q9fF#*7!D9%d7Tg5RW8aLy57j@XdQ=mmg3&ANH>IzQUrzoMr;V|MRPn}?pGSxxV(kNw_l2Qro9Gh!AS}*X56d`z&tQd(!PUJ|p=|znE#zXe<h(=C^>u##?VOLvR?8<JrP7cQ%DA3mtt<gpO)wJMTh7$&3{Z3hdOXuz-GQR2}&#I#4mE(&TJkld``eJqYv2nz&duMMH5tT_u#SZ4C1|lTHwpDLk>J_d-F-p*Gvr~bh>pRzEVq7lljhieU;hZy<u`;_}AT2F%D#!FwHFF*YliplwRQ)M_>Knnnw`0pwehL??_N}ca>p6@PEFoWyEi3$bT$_j$xiP66sJoFV1H(qTT~9B_b867_;`Qwm<vhofsh~!-@UxIfv&B+x>9)uX&>fhiW<656MG1xM3Y*m;hz9LqVw(q}JtS7U<GBib;~KOO%q<8tlC36nhX8)wHF`)-mWCVakWPRI>F7}|O5Ntwyu!OSxL7Xg2bso5DzGm|ZIhTQY069hk*=FhhOOIBWudlOB67f?(Bq^<JSUuxIDgL1lJlz)L=xvF3s#%E^sq3YhNF_cN+@ZK=j?l{=`Mu|w8GdqL=FyXO05|@PDt6#v30S>LwPy6R-9O}JS*jk+zze*vfdrflc)W$sf>DN?E(!q<F3E1x3k67StL70m`1fhdC;@`+@_-FnOP*(hZ9cAB`h>)IZp2a^F3ohvRdpu7Thh2NFCiuWDj0B<^k&RJi}E`K6-G4miun}R$=Fj#Gy;g$I4g&I*B7zJ1T@?r@&!}h!$z4uG6dNSc!U<*d?KksMD#Sl%mstE+p};KM}KPdEQZxTI}Xu_i;7!-)ci$W@rUTCU<w+<JY6pzQ)I~I1*7)Ce!NnD$(ZH@*E+uQSm4j`f@Z^b|t+rUmv*b^hReA+r)NnLrixBGPeO!tKP|3uHucR&LS#z)$=8bpgW^<n(<0A>g^Im+BlSmv0J_)6-!ln)?n3-lTiKI4YcHTH5+F8qIY#fb+fIVxKlr@N855uvxp5#Hj|s?ZZoNPd!)#BCLOB|!i{YuLZGDqqse2xC(p;RRjNuSYY?Fax><}&lF{}?jA2`20w&fSmXYi@N@Iy#4jgfzpaZGMZE#`ODhzJ>#Ewj9Xiz?^WN?3lBa|}XW3rXT`vxO2jUJI~?xB3Npa5rwH%C#J+NCKG<LXB%o^BS47=wt?F>&UAcz)kVwJR2}b-nFfb0&yX?&3X6CfD1Z*_mqeg-%3>E~IzFKu6f<q=&Ea@rL7Mid_ZYUxWZZ`qx=uF}=nM!M_mX-|v4t?|X6r4Hy348zY_`Q{Bhk-^2gvQk{P9U*BK;)ulV_$8UB#s!qwU*_c1xru}~BU$1A_adk|EXD)xdgg?puyj;HNg6r|FC)wZr^O8t?fBClYrJvu!$II>Z{dp+$|Mt-B4}MBf^&hk!WN+V}h;HwjBWlp8&O;b#@Q?}pgJ)a)0Y4b^>s)Qjez5$FthTAqJj4xuqngccA*pGLs*b-QuiSkb(2GU&dH4zQ(8vR5-0xP${a?Y!5!b7=cy`bq|B5qw_@T*<BlGI07c$x(QO^?(?SA|IzDd_8zBZz&-5)G`LJ=Z-XXq!csWD%xHeNB-s-scJ&IeAm+wYHCH1&@t-+>#|QI{IAe78DkQ?D|82=k8kvpT*Cc>CjzKQ@y55|%j+J_x!meSguU;QLFsDEPWp>RG0Q{O3tZm0wg>HC<0IMd=-Qql(_!{^Q4yJl`>2o}2%E;QWjXd4h$jp5%lE4j%;1bKWt13*SQqp9J6OiGr>RYh8k05B*B$b^SL~e(Zmy@fGe(^vC~v;RVrCEfaqI^+xCQCFJx05yqrsKOWv-UMwec=P$%AAKZPD1;>z_&>FhxdT%G5`gq?#9_Ro3HmI`fx4+(O`8>H5%#9l!{PTIzayFr^K7+o<)??wZteP)hP`@61#rzlY*C(Wg;Ao+CzhM0*OmFZ%q55zZR}w-bYxhl0Z{nw(9%6d%)0x8U(5AL71fjIwUy@H;;lHI)2q@{U^pbo>_fW|z`~&eD(#N73>i)^_CydYa5MjvH8+V_1c<%oBK=apwo5#Zo{-R4J{M<oaDPKM$5p-2C`KRLE6!87!hhQHL@H-!QLP#fD5<O}2U^FE1<nxm}KQx7We>d$ftUs6ECH;GQzQ@Fa&T|I|C3~m;BQTy@|CUakWNzR;-`(bEYyM$pM=%u${@^?D2_N!C<dug%-{}7pvi83@;}ibR|GeY=>#xt2c}4oJcTd$NK2*&^Ro`E1%T+(co#zBU<T-tR2|^AYe)SIaXEKbKuPdoH);|$^HO4D}Zx4E3RsU!3*Po}q-i#<pw)X<l=0-YEj$=9BU&etgy**AIx#Ri#YySDx>DSs+{o99Jjvz`8N3ooDcX?FF?=R1#5BWl%hgZG*pSnK{)r1Y<1V2Q2fFqB1`TpurZ!kz`sTA_pYu0&pg(uQSM1AC&pZR;*?Md$+vd<SU`~>`qzy9$v=2we9+4$=RL-=POeFoeQQg3=XO45dRBwJ8F^z;mv_ZWO?;pZUyyI_5SJwv_pDdJv>jbJ=A{i7jyO%CrGe9(C)F!>Ph`GP+k>v8T?2|qfY?_sBdd~$&|++UlcS8IQJuy=Uth&~Sd!o{NqhDbg*{qqy5$L8bVBN;x#_$gC9_ugaRVZ1*-`~17DzrVzOB+<9MUsKVmN_<oG-VE}@_nXS~{ZrX`!~73L=);5u=}p>yyOQ@-g5h^PeSenHS5d=*PdEDha7EL7Grwy7$d3o<->N4WF8tXV>LYkx9+v!FV7$8F!?Le|_!56f{WvPWoi4v-@y|;5MB#@d|F$g~eq-`Szw-VgTBfI(zVxvuDAJoT!yta}@JX;gzlHsj<9+*-zaIQQPK1T>EAh{K{rTkhulKU?1!G7eO#ag5Gg3(p(WYPU(i1oSD8liVUt972h2Y;MxkrY2PjsK?y3+rq!Vl{|34MS0Nbb*6{}c#$xPfH)p+yBxg3qe`0ThPB7qfk`+nY1|@P!Y1KLh{GGam1WuTJrLP&MEEH+(TI?{gFU&e->-6_5{zx8?u*vtPWnOHatKq`jRC`*)I~J9R#cu`w6U9AUJE_UVPQ!+$~dsBus8{Jyq+Ci{~Zj}vdL{6mDt>uWG;p<cpR214aMT;<z;;owzI@Acr38eg6JBQ1V-=38U@Vb~u@^7lUX=eIiD{Px!$U;2|bdp`ERoJu}x(U(;Ar3(E`lk){f>OWtq-QU-q_a4>>)AqwDU$gU%^7Z?=^n`sZzt8CZTUmM@`)_sW9rtew)8EeRKdRJMmY$g_MEHa4zpYXs<H<*A`H9mHlCO-0qCDPy;`h%VV1Ir8wDhF#;PkEQd9%kWOYcpWbu{^nWQ4}yP3!n-f=7k;Vu)9Om(tsu5^g{Jx<QS`bcbG0qi-*d{in8}mvr`1c1V7)3-pqGoJ{`GY`s!=ZSg{u9_IZqw>(x}>)fXv?vd`?t!00dx95QeXn4XmbxYL3KIsARwOIckZ9CS+7rjTZ{(6J*)cxaRm`RZ@!T%IJB=~;xYf!(bDP9Vz#oM&;4)}R^Oa1Yy;P0W}hl>)w_k}P$c)?o({Gj?i+C2f@OX7PaeoISF4|;>ZzqY`?C_MSaC42+^0C;rM$=4a-IiCEP{_B$xz7D=0ZTKtIzkWLce^%gwskae2>=-gnw|S)!W}ZjW!BU;hw@=Hd=kh}XPc$D%!&XJjx5GZzm7Ih0kyk#c>TT=M*ZlDd=l}Qa>_1=p;zvPvk_a8=aX#$Bd|Q{oz<mhxRrd6+vJdE5n;ZOI`mdh$Wz73w*;j!>|NE^Cf2aRX4*Ah`yp44KZH)U3*)N{`w*GD~=@cqNHI*=%>-z8Q^?&olf4GGC|HT6zBUk|cn7YEO`Xy<;-RS&7HU~;SrSq@z{xhZc&vN8?4*aUB&nkIO6Uggj!fV=oUwr7~(U$zgz-t~w-lu@u|9DyRs?bL_@z+N%Il}S1!~4D7e+c$!rMHGVJokUL$V0eLHIKFM_n+_Uf4!7H7nfI^?tRI9y=VGocY1|>r0Z8@Jgyc#PJjPO2l*QsiP8@gUu(k`F8>xRRL2iwc<E<he%m^JC^!`7rS#+crzm<@<==vBKQ6XjRsE%rdo$Z#PZd2L|NKuX`PU!+UptJi_f@hJGR!-^`MLS~n}B&#v5)UH{}dOtqaNPcz)v@(uZzF53hwr4{U_ubObGp0{RPyp4o}$2{~8$Yf%ovp-*6uG@X9l8{;9No4u-b__*(<}UmLVv9sU>paJ{e>`s)4sYlri2>sMUwcg&x!0^iH$Ki1mEDEtqp{R7=cT6&GES24f8^YqGM;+^Q%rt9PAukrAUtUvP9|F_%vAH(;XOd%>$I(hKh4el3AZ^Zs@MFsi!9wbEfL0om8A@KnC&re1A72-AIKUwzE2J$e@yYjx?W<2<P<^C~Kq<+%s*L?BkEA!ufsxN>2(?!K^$J_ssX+Fa5&j+C=#N+L0rw=FnNuW@aC(u7kL_hb+UxB`T(D~a0CMc5m`IF6;-s^SRe0jfx{v2MfIq9w5f5g}CuBLy|(j(;l?vnRQ9{T9MA4DEa>oX(#GR*#N!2S)%zwc*$1N=WE^`Eo%Ki@&Wb)D4nSGe^<#lK$Qe)=@@Gm*;|H~jG}@x8PE{&oN3i{Y;yd;Z&<?@yS1d?x&~bNl1}0UJuh%>')).decode("utf-8"),"<embedded-ramesh>","exec"),_BASE_NS)
_MAX_SWITCHES=5
_STATE={0:{},1:{}}

def _get(value,key,default=None):
 if isinstance(value,dict): return value.get(key,default)
 getter=getattr(value,"get",None)
 return getter(key,default) if callable(getter) else getattr(value,key,default)

def _seat(obs): return 1 if int(_get(obs,"player",0) or 0)==1 else 0

def _farm(obs,seat):
 farms=list(_get(obs,"farms",[]) or [])
 return farms[seat] if seat<len(farms) else {}

def _tile(farm,position):
 try:
  x,y=int(position[0]),int(position[1])
  return (_get(farm,"tiles",[]) or [])[y][x]
 except (IndexError,TypeError,ValueError): return None

def _reset(step):
 return {"last":step,"selected":0,"pickup_debt":0,"place_debt":0,
         "tracked":[],"extra_produced":0,"extra_sold":0}

def _rebalance(obs,action,step):
 seat=_seat(obs); state=_STATE[seat]
 if step==0 or not state or step<int(state.get("last",-1)):
  state=_reset(step); _STATE[seat]=state
 state["last"]=step
 action=copy.deepcopy(action or {})
 action={"farmer":list(action.get("farmer") or ["PASS"]),
         "hands":[list(order or ["PASS"]) for order in (action.get("hands") or [])],
         "market":[list(order) for order in (action.get("market") or [])]}
 market=action["market"]
 for order in market:
  if not (isinstance(order,list) and len(order)>=3 and order[:2]==["BUY_ANIMAL","COW"]):
   continue
  quantity=max(0,int(order[2] or 0)); remaining=_MAX_SWITCHES-int(state["selected"])
  if quantity<=0 or quantity>remaining: continue
  order[1]="SHEEP"; state["selected"]+=quantity
  state["pickup_debt"]+=quantity; state["place_debt"]+=quantity

 farm=_farm(obs,seat)
 positions=[_get(farm,"farmer"),*list(_get(farm,"hands",[]) or [])]
 actors=[action["farmer"],*action["hands"]]
 private=_get(obs,"private",{}) or {}
 shed=_get(private,"shed",{}) or {}
 inventories=list(_get(private,"inventories",[]) or [])
 tracked=state["tracked"]
 for index,(order,position) in enumerate(zip(actors,positions)):
  inventory=inventories[index] if index<len(inventories) else {}
  if (state["pickup_debt"]>0 and isinstance(order,list) and len(order)>=3
      and order[:2]==["PICKUP","COW"]):
   quantity=max(0,int(order[2] or 0))
   if quantity and quantity<=state["pickup_debt"] and int(_get(shed,"SHEEP",0) or 0)>=quantity:
    order[1]="SHEEP"; state["pickup_debt"]-=quantity
  before=_tile(farm,position)
  if (state["place_debt"]>0 and isinstance(order,list) and len(order)>=2
      and order[:2]==["PLACE","COW"] and isinstance(before,dict)
      and before.get("kind")=="PASTURE" and "animal" not in before
      and int(_get(inventory,"SHEEP",0) or 0)>0):
   order[1]="SHEEP"; state["place_debt"]-=1
   key=[int(position[0]),int(position[1])]
   if key not in tracked: tracked.append(key)
  if (isinstance(order,list) and order and order[0]=="HARVEST"
      and [int(position[0]),int(position[1])] in tracked
      and isinstance(before,dict) and before.get("animal")=="SHEEP"):
   state["extra_produced"]+=max(0,int(before.get("yield_units",0) or 0))

 planned=sum(max(0,int(order[2] or 0)) for order in market
             if isinstance(order,list) and len(order)>=3 and order[:2]==["SELL","WOOL"])
 unsold=max(0,int(state["extra_produced"])-int(state["extra_sold"]))
 available=max(0,int(_get(shed,"WOOL",0) or 0)-planned)
 prices=_get(_get(obs,"market",{}) or {},"prices",{}) or {}
 price=int(_get(prices,"WOOL",0) or 0)
 quantity=min(unsold,available)
 if quantity>0 and (planned>0 or price>=180 or step>=680):
  existing=next((order for order in market if isinstance(order,list) and len(order)>=3
                 and order[:2]==["SELL","WOOL"]),None)
  if existing is not None: existing[2]=int(existing[2] or 0)+quantity
  elif len(market)<10: market.append(["SELL","WOOL",quantity])
  else: quantity=0
  state["extra_sold"]+=quantity
 action["farmer"],action["hands"],action["market"]=actors[0],actors[1:],market[:10]
 return action

def agent(obs,configuration=None):
 action=_BASE_NS["agent"](obs)
 step=max(0,int(_get(obs,"step",0) or 0))
 shops=list(_get(_get(obs,"town",{}) or {},"unlocked_shops",[]) or [])
 if 72<=step<712 and shops and shops[0]=="YARN_STORE":
  return _rebalance(obs,action,step)
 return action
''')


### Market and inventory support


In [8]:
SOURCE_PARTS.append(r'''

# E156 market adapter; original whole controller retained.
import copy, math
_E156_BASE = agent
_YARN_REVERT = {}
_BRUNCH_REVERT = {}
_E156_CONFIG = {'economy': False, 'priority': True, 'coherent': False, 'productive': False}
def _get(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)

def _copy_action(action):
    action = copy.deepcopy(action or {})
    return {
        "farmer": list(action.get("farmer") or ["PASS"]),
        "hands": [list(order or ["PASS"]) for order in (action.get("hands") or [])],
        "market": [list(order) for order in (action.get("market") or [])],
    }

def _seat(obs):
    return 1 if int(_get(obs, "player", 0) or 0) == 1 else 0

def _farm(obs, seat):
    farms = list(_get(obs, "farms", []) or [])
    return farms[seat] if seat < len(farms) else {}

def _align_hands(action, obs):
    action = _copy_action(action)
    expected = len(_get(_farm(obs, _seat(obs)), "hands", []) or [])
    hands = list(action.get("hands") or [])
    if len(hands) < expected:
        hands.extend([["PASS"] for _ in range(expected - len(hands))])
    action["hands"] = [list(order or ["PASS"]) for order in hands[:expected]]
    return action

def _tile_at(farm, position):
    try:
        x, y = int(position[0]), int(position[1])
        return (_get(farm, "tiles", []) or [])[y][x]
    except (IndexError, TypeError, ValueError):
        return "LOCKED"

def _existing_sell(action, item):
    return sum(
        max(0, int(order[2]))
        for order in (action.get("market") or [])
        if len(order) >= 3 and order[0] == "SELL" and order[1] == item
    )

def _e154_step(obs, configuration=None):
    # The official callback supplies step to both seats. The fallback also
    # supports replay observations, in which shared fields may be omitted.
    value = _get(obs, "step", None)
    if value is not None:
        return max(0, int(value))
    turns = int(_get(configuration or {}, "turnsPerDay", 24) or 24)
    return max(0, int(_get(obs, "day", 0) or 0) * turns
               + int(_get(obs, "hour", 0) or 0))

_FR_ITEMS = ('MELON', 'MILK', 'STRAWBERRY', 'WOOL')

_SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}

_E155_PARAMS = {
    "WHEAT": (25, 400, "sqrt", .80, "log", .20),
    "CARROT": (35, 450, "hinge", 1., "sqrt", .70),
    "TOMATO": (60, 200, "hinge", .40, "sqrt", .60),
    "STRAWBERRY": (120, 100, "sqrt", .70, "linear", 1.60),
    "MELON": (250, 300, "log", .20, "sq", 3.60),
    "EGG": (50, 332, "hinge", .40, "log", .20),
    "MILK": (160, 122, "sqrt", .60, "linear", 1.60),
    "WOOL": (200, 105, "log", .20, "sq", 3.20),
    "FERTILIZER": (100, 200, "linear", .40, "linear", .40),
}

def _e155_shape(name, x, t):
    x = max(0., x)
    if name == "sq": return x * x
    if name == "sqrt": return math.sqrt(x)
    if name == "log": return math.log1p(x)
    if name == "log10": return math.log10(1. + x)
    if name == "hinge":
        u = x / t if t > 0 else x
        return u + 8. * max(0., u - 1.) ** 2 if t > 0 else x
    return x

def _e155_price(item, inventory, market=None):
    base, t, below, bt, above, at = _E155_PARAMS[item]
    p = _get(_get(market or {}, "params", {}) or {}, item, {}) or {}
    base, t = p.get("base", base), p.get("T", t)
    i0 = p.get("I0", 10000)
    if inventory < i0:
        shape, target, sign = p.get("below_func", below), p.get("below_target", bt), 1
    else:
        shape, target, sign = p.get("above_func", above), p.get("above_target", at), -1
    value = base + sign * target * base * _e155_shape(shape, abs(inventory - i0), t) / _e155_shape(shape, t, t)
    return max(1, int(round(value)))

def _e155_demand(obs, item, start, end, configuration=None):
    # Future shop draws are not observations: count ONLY visible instances.
    cfg = configuration or {}
    center = max(1, int(_get(cfg, "townCenterSellInterval", 24)))
    interval = max(1, int(_get(cfg, "townShopSellInterval", 4)))
    amount = 0
    town = _get(obs, "town", {}) or {}
    weight = sum(2 if len(_SHOP_PRODUCTS.get(shop, ())) == 1 else 1
                 for shop in (_get(town, "unlocked_shops", []) or [])
                 if item in _SHOP_PRODUCTS.get(shop, ()))
    for t in range(start, end):
        if item != "FERTILIZER" and t % center == 0: amount += 1
        if t % interval == 0: amount += weight
    return amount

def _e155_post_field(obs, action, configuration=None):
    """Predict shed deposits/pickups before market, with real capacity semantics."""
    farm = _farm(obs, _seat(obs))
    private = _get(obs, "private", {}) or {}
    shed = dict(_get(private, "shed", {}) or {})
    invs = [dict(v) for v in (_get(private, "inventories", []) or [])]
    board = int(_get(configuration or {}, "boardSize", 10))
    cap = int(_get(configuration or {}, "shedCapacity", 100))
    middle = {board // 2 - 1, board // 2}
    positions = [_get(farm, "farmer", [0, 0]), *(_get(farm, "hands", []) or [])]
    for i, (pos, order) in enumerate(zip(positions, [action["farmer"], *action["hands"]])):
        if i >= len(invs) or not order: continue
        inv = invs[i]
        adjacent = pos[0] in middle and pos[1] in middle
        op = order[0]
        item = order[1] if len(order) > 1 else None
        qty = max(0, int(order[2])) if len(order) > 2 else 1
        tile = _tile_at(farm, pos)
        if op == "PICKUP" and adjacent:
            n = min(qty, shed.get(item, 0))
            shed[item] = shed.get(item, 0) - n
            inv[item] = inv.get(item, 0) + n
        elif op == "DROP" and adjacent:
            for product, n in list(inv.items()):
                take = min(n, max(0, cap - sum(shed.values())))
                shed[product] = shed.get(product, 0) + take
            inv.clear()
        elif op == "PLACE":
            animal_place = (item in ("COW", "SHEEP", "GOOSE") and isinstance(tile, dict)
                            and tile.get("kind") == ("COOP" if item == "GOOSE" else "PASTURE")
                            and not tile.get("animal"))
            if animal_place:
                inv[item] = max(0, inv.get(item, 0) - 1)
            elif adjacent:
                take = min(qty, inv.get(item, 0), max(0, cap - sum(shed.values())))
                shed[item] = shed.get(item, 0) + take
                inv[item] = inv.get(item, 0) - take
        elif op == "HARVEST" and isinstance(tile, dict):
            product = tile.get("crop") or {"COW": "MILK", "SHEEP": "WOOL", "GOOSE": "EGG"}.get(tile.get("animal"))
            if product: inv[product] = inv.get(product, 0) + max(0, int(tile.get("yield_units", 0)))
    return shed, sum(sum(v.values()) for v in invs)

def _e155_market(obs, action, step, configuration=None):
    _E155_TELEMETRY["economic_turns"] += 1
    market = _get(obs, "market", {}) or {}
    supply = _get(market, "inventory", {}) or {}
    shed, carried = _e155_post_field(obs, action, configuration)
    cap = int(_get(configuration or {}, "shedCapacity", 100))
    turns = int(_get(configuration or {}, "turnsPerDay", 24))
    terminal = step >= int(_get(configuration or {}, "episodeSteps", 720)) - 2
    cash = float(_get(_farm(obs, _seat(obs)), "money", 0))
    safe_cash = cash >= _e155_cash_reserve(obs, step)
    # Keep room for planned purchased feed and the next automatic daily drop.
    buys = sum(int(o[2]) for o in action["market"] if len(o) > 2 and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL"))
    room_needed = max(0, sum(shed.values()) + buys + 12 + (carried if step % turns >= turns - 4 else 0) - cap)
    products = _FR_ITEMS + (("CARROT",) if _E155_CONFIG.get("crops") else ())
    quantities = {}
    for item in products:
        stock = max(0, int(shed.get(item, 0)))
        if stock <= 0: continue
        inventory = supply.get(item, 10000)
        q = 0
        # Price impact is per-unit; $1 units do not increase shared supply.
        while q < stock and _e155_price(item, inventory + q, market) > 1:
            q += 1
        requested = _existing_sell(action, item)
        if _E155_CONFIG["mode"] == "scheduled":
            q = min(q, requested)
            # Unsold remainder can re-enter when a visible demand pulse recovers price.
            if requested == 0 and _e155_price(item, inventory, market) > 1:
                q = min(stock, q if q else 1)
        demand_now = _e155_demand(obs, item, step, step + 1, configuration)
        if (_E155_CONFIG["pulse"] and demand_now and safe_cash and room_needed == 0
                and step % turns < turns - 4 and not terminal):
            # Reconsider after one observed consumption event, never a fixed sell calendar.
            _E155_TELEMETRY["pulse_holds"] += q
            q = 0
        if q < stock and _e155_demand(obs, item, step, min(step + turns, 719), configuration) <= 0:
            # There is no known recovery opportunity: do not hoard worthless units.
            q = stock
        if not safe_cash and requested > q: q = min(stock, requested)
        quantities[item] = stock if terminal else q
    # Storage pressure is resolved by the lowest-value stock first, preserving
    # high-value optionality; all sales still obey the ten-order limit below.
    remaining = max(0, room_needed - sum(quantities.values()))
    for item in sorted(quantities, key=lambda k: _e155_price(k, supply.get(k, 10000), market)):
        if remaining <= 0: break
        take = min(remaining, shed[item] - quantities[item])
        quantities[item] += take
        remaining -= take
        _E155_TELEMETRY["forced_units"] += take
    _E155_TELEMETRY["floor_held_units"] += sum(max(0, shed.get(k, 0) - q) for k, q in quantities.items()
                                                if _e155_price(k, supply.get(k, 10000), market) == 1)
    # Preserve order positions (including intentional zero placeholders), so
    # moving one premium order does not silently reschedule procurement.
    result, emitted = [], set()
    for raw in action["market"]:
        order = list(raw)
        if len(order) > 2 and order[0] == "SELL" and order[1] in products:
            item = order[1]
            order[2] = quantities.get(item, 0) if item not in emitted else 0
            emitted.add(item)
        result.append(order)
    for item in sorted(quantities, key=lambda k: quantities[k] * _e155_price(k, supply.get(k, 10000), market), reverse=True):
        if item not in emitted and quantities[item] > 0 and len(result) < 10:
            result.append(["SELL", item, quantities[item]])
    action["market"] = result[:10]
    return action

def _e156_ready(farm, item):
    amount = 0
    for row in _get(farm, "tiles", []) or []:
        for tile in row:
            if not isinstance(tile, dict): continue
            product = tile.get("crop") or {"COW": "MILK", "SHEEP": "WOOL"}.get(tile.get("animal"))
            if product == item: amount += max(0, int(tile.get("yield_units", 0)))
    return amount

def _e156_priority(obs, action):
    market = _get(obs, "market", {}) or {}
    inventory = _get(market, "inventory", {}) or {}
    enemy = _farm(obs, 1 - _seat(obs))
    sales, rest = [], []
    for order in action["market"]:
        if len(order) > 2 and order[0] == "SELL" and order[1] in _FR_ITEMS and order[2] > 0:
            sales.append(order)
        else:
            rest.append(order)
    def urgency(order):
        item, n = order[1], max(0, int(order[2]))
        stock = inventory.get(item, 10000)
        drop = _e155_price(item, stock, market) - _e155_price(item, stock + n, market)
        # The public harvestable rival stock is a risk signal, not hidden stock.
        return (drop * (1 + _e156_ready(enemy, item)), n * _e155_price(item, stock, market))
    sales.sort(key=urgency, reverse=True)
    result = sales + rest
    _E156_TELEMETRY["reordered_turns"] += int(result != action["market"])
    action["market"] = result[:10]
    return action


_E156_ACTION = {}
_E156_OBSERVATION = None
_E156_TELEMETRY = {"exceptions": 0, "reordered_turns": 0, "economy_turns": 0,
                    "yarn_handoff": 0, "brunch_handoff": 0,
                    "coherent_trace_changes": 0, "coherent_future_changes": 0}
_E155_CONFIG = {"mode": "inventory", "pulse": False, "crops": False}
_E155_TELEMETRY = {"economic_turns": 0, "pulse_holds": 0, "floor_held_units": 0, "forced_units": 0}


def _e156_trace_actor_action(step, actor):
    route = _V174_NS["_v154_route"](_E156_OBSERVATION, step)
    action = route[min(max(0, int(step)), len(route) - 1)] or {}
    if actor == "farmer": result = list(action.get("farmer") or ["PASS"])
    else:
        hands = action.get("hands", []) or []
        result = list(hands[actor] if actor < len(hands) else ["PASS"])
    _E156_TELEMETRY["coherent_trace_changes"] += int(result != _E156_BASE_TRACE(step, actor))
    return result


def _e156_future_quantity(step, item):
    route = _V174_NS["_v154_route"](_E156_OBSERVATION, step + 1)
    if not 0 <= step + 1 < len(route): return 0
    result = sum(max(0, int(o[2])) for o in route[step + 1].get("market", [])
                 if len(o) > 2 and o[:2] == ["SELL", item])
    _E156_TELEMETRY["coherent_future_changes"] += int(result != _E156_BASE_FUTURE(step, item))
    return result


if _E156_CONFIG.get("coherent"):
    _E156_BASE_TRACE = _V174_NS["_trace_actor_action"]
    _E156_BASE_FUTURE = _V174_NS["_future_quantity"]
    _V174_NS["_trace_actor_action"] = _e156_trace_actor_action
    _V174_NS["_future_quantity"] = _e156_future_quantity


def _e156_productive_priority(obs, action):
    # Floor-price sales cannot depress a rival's price. Never let them jump
    # ahead of productive fertilizer/premium sales. Feed orders keep their
    # mutual order; no quantities or farming actions are changed.
    market = _get(obs, "market", {}) or {}
    inventory = _get(market, "inventory", {}) or {}
    enemy = _farm(obs, 1 - _seat(obs))
    sales, rest = [], []
    for order in action["market"]:
        if (len(order) > 2 and order[0] == "SELL" and order[1] in (*_FR_ITEMS, "FERTILIZER")
                and order[2] > 0 and _e155_price(order[1], inventory.get(order[1], 10000), market) > 1):
            sales.append(order)
        else:
            rest.append(order)
    def urgency(order):
        item, n = order[1], max(0, int(order[2]))
        stock = inventory.get(item, 10000)
        quote = _e155_price(item, stock, market)
        risk = _e156_ready(enemy, item)
        if item == "FERTILIZER":
            risk = sum(bool(t.get("fertilizer_available")) for row in _get(enemy, "tiles", [])
                       for t in row if isinstance(t, dict) and t.get("animal"))
        return ((quote - _e155_price(item, stock + n, market)) * (1 + risk), n * quote)
    sales.sort(key=urgency, reverse=True)
    result = sales + rest
    _E156_TELEMETRY["reordered_turns"] += int(result != action["market"])
    action["market"] = result[:10]
    return action


def _e155_cash_reserve(obs, step):
    # All purchases are retained. In the new architecture future route actions
    # are not known, so reserve current commitments plus next-day working cash.
    prices = _get(_get(obs, "market", {}), "prices", {})
    cash = 0
    hires = int(_get(_farm(obs, _seat(obs)), "hires_today", 0))
    for order in _E156_ACTION.get("market", []):
        op = order[0]
        if op == "HIRE":
            a, b = 1, 1
            for _ in range(hires): a, b = b, a + b
            cash += a
            hires += 1
        elif op == "BUY_LAND": cash += 2000
        elif len(order) > 2:
            item, qty = order[1:3]
            if op == "BUY_PRODUCT": cash += prices.get(item, 100) * qty
            elif op == "BUY_SEED": cash += {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}.get(item, 100) * qty
            elif op == "BUY_ANIMAL": cash += {"COW": 400, "SHEEP": 500, "GOOSE": 300}.get(item, 500) * qty
    return max(3500 if step < 384 else 1800, cash + 1000)


def agent(obs, configuration=None):
    global _E156_ACTION, _E156_OBSERVATION
    step = min(_e154_step(obs, configuration), 719)
    # Both nested controllers observe the normalized legal clock. No future
    # route is evaluated here, and no hidden seed/identity is inspected.
    normalized = dict(obs)
    normalized["step"] = step
    _E156_OBSERVATION = normalized
    action = _align_hands(_E156_BASE(normalized, configuration), normalized)
    fallback = _copy_action(action)
    _E156_ACTION = action
    _E156_TELEMETRY["yarn_handoff"] = int(any(_YARN_REVERT.values()))
    _E156_TELEMETRY["brunch_handoff"] = int(any(_BRUNCH_REVERT.values()))
    try:
        if step >= 192:
            if _E156_CONFIG["economy"]:
                action = _e155_market(normalized, action, step, configuration)
                _E156_TELEMETRY["economy_turns"] += 1
            if _E156_CONFIG.get("productive"):
                action = _e156_productive_priority(normalized, action)
            elif _E156_CONFIG["priority"]:
                action = _e156_priority(normalized, action)
        return _align_hands(action, normalized)
    except Exception:
        _E156_TELEMETRY["exceptions"] += 1
        return fallback


def e156_kaggle_entrypoint(obs, configuration=None):
    return agent(obs, configuration)


agent.telemetry = _E156_TELEMETRY
e156_kaggle_entrypoint.telemetry = _E156_TELEMETRY

def _e155_idle_economy(obs, action, step):
    # Keep all movements, planting and procurement. Only replace proven no-ops.
    farm = _farm(obs, _seat(obs))
    private = _get(obs, "private", {}) or {}
    invs = _get(private, "inventories", []) or []
    positions = [_get(farm, "farmer", [0, 0]), *(_get(farm, "hands", []) or [])]
    orders = [action["farmer"], *action["hands"]]
    day = step // 24
    seen = set()
    for i, (pos, order) in enumerate(zip(positions, orders)):
        if i >= len(invs) or tuple(pos) in seen: continue
        seen.add(tuple(pos))
        tile = _tile_at(farm, pos)
        if not isinstance(tile, dict): continue
        op = order[0]
        free = (op == "PASS" or (op == "WATER" and tile.get("watered_today"))
                or (op == "CARE" and tile.get("cared_today"))
                or (op == "FEED" and tile.get("fed_today"))
                or (op == "COLLECT_FERTILIZER" and not tile.get("fertilizer_available"))
                or (op == "HARVEST" and not tile.get("yield_units")))
        if not free: continue
        crop, animal = tile.get("crop"), tile.get("animal")
        # Harvesting ongoing production cannot destroy the crop/animal. Earlier
        # collection also prevents loss at the animal's six-unit holding cap.
        if (animal or crop in ("STRAWBERRY", "TOMATO")) and tile.get("yield_units", 0) > 0:
            orders[i] = ["HARVEST"]
            _E155_TELEMETRY["extra_harvests"] += 1
        elif (crop and invs[i].get("FERTILIZER", 0) > 0 and tile.get("watered_today")
              and day > tile.get("fertilized_until_day", -1) and day < 28):
            age = day - int(tile.get("planted_day", day))
            if crop in ("STRAWBERRY", "TOMATO"):
                dates = (10, 12, 14, 16) if crop == "STRAWBERRY" else (8, 9, 10, 11)
                extra = sum(age < d <= age + 3 and day + d - age < 30 for d in dates)
            else:
                low, high, cap = {"WHEAT": (2, 4, 6), "CARROT": (2, 3, 4), "MELON": (6, 12, 6)}[crop]
                # Today's WATER has already happened; fertilizer cannot change it.
                extra = min(max(0, cap - tile.get("yield_units", 0)), sum(low <= age + d <= high for d in (1, 2)))
            market = _get(obs, "market", {}) or {}
            inv = _get(market, "inventory", {}) or {}
            value = extra * _e155_price(crop, inv.get(crop, 10000) + extra, market)
            cost = _e155_price("FERTILIZER", inv.get("FERTILIZER", 10000), market)
            if extra > 0 and value > cost + 10:
                orders[i] = ["FERTILIZE"]
                _E155_TELEMETRY["extra_fertilizations"] += 1
        elif animal and tile.get("fed_today") and not tile.get("cared_today"):
            first, period = {"COW": (8, 2), "SHEEP": (6, 3), "GOOSE": (4, 1)}[animal]
            age = day - int(tile.get("placed_day", day))
            next_age = first if age < first else first + ((age - first) // period + 1) * period
            # Tonight's production consumes OLD care, then banks today's care.
            if next_age == age + 1: next_age += period
            if day + next_age - age < 30:
                orders[i] = ["CARE"]
                _E155_TELEMETRY["extra_care"] += 1
    action["farmer"], action["hands"] = orders[0], orders[1:]
    return action

''')


### Livestock valuation and compatible allocation


In [9]:
SOURCE_PARTS.append(r'''# E157: observable-state allocation at compatible scheduled investment slots.
_E157_CONFIG = {'allocate': True, 'inventory': False, 'service': False, 'terminal': True, 'geese': True, 'coherent': True, 'future_weight': 0.35, 'hysteresis': 300.0, 'goose_limit': 2}
_E157_PARENT_BASE = _E156_BASE
_E157_PARENT_ENTRY = agent  # Capture the body, not a forwarder through global agent.
_E157_STATE = {}
_E157_TELEMETRY = {"exceptions": 0, "allocation_decisions": 0, "converted_units": 0,
                   "converted_pickups": 0, "converted_placements": 0,
                   "inventory_turns": 0, "terminal_sales": 0,
                   "extra_harvests": 0, "extra_fertilizations": 0, "extra_care": 0,
                   "pending_pickups": 0, "pending_placements": 0,
                   "deferred_seed_orders": 0, "pending_market_orders": 0,
                   "first_divergence": -1}
_E157_ANIMALS = {"COW": ("MILK", 8, 2, 400), "SHEEP": ("WOOL", 6, 3, 500),
                 "GOOSE": ("EGG", 4, 1, 300)}

def _e157_state(obs, step):
    seat = _seat(obs)
    state = _E157_STATE.get(seat)
    if state is None or step == 0 or step < state['step']:
        state = {'step': step, 'queues': {'COW': [], 'SHEEP': []},
                 'carrying': {}, 'changed': False, 'decisions': [], 'deferred': []}
        _E157_STATE[seat] = state
    state['step'] = step
    return state

def _e157_herd(farm):
    return [dict(tile) for row in _get(farm, 'tiles', []) for tile in row
            if isinstance(tile, dict) and tile.get('animal') in _E157_ANIMALS]

def _e157_production(herd, day, end):
    production = {p: [0.] * end for p in ('MILK', 'WOOL', 'EGG')}
    for tile in herd:
        product, first, period, _ = _E157_ANIMALS[tile['animal']]
        placed = int(tile.get('placed_day', day))
        pending = float(tile.get('pending_care_bonus', 0))
        for future in range(day + 1, end):
            if future >= placed + first and (future - placed - first) % period == 0:
                production[product][future] += min(4. if tile['animal'] == 'GOOSE' else 6., 1. + pending)
                pending = 0.
            pending += 1.  # Production consumes OLD care before today's care.
    return production

def _e157_projection(obs, step, extra=None, quantity=0):
    """Finite-horizon own revenue INCLUDING cannibalization of the current herd.

    This is an approximate allocation model, not an exact engine simulator.
    Future shop identities and opponent private stock are never inspected.
    """
    day, end = step // 24, 30
    own = _e157_herd(_farm(obs, _seat(obs)))
    enemy = _e157_herd(_farm(obs, 1 - _seat(obs)))
    private = _get(obs, 'private', {}) or {}
    stored = [private.get('shed', {}), *(private.get('inventories', []) or [])]
    for animal in _E157_ANIMALS:
        own.extend({'animal': animal, 'placed_day': day + 1} for _ in
                   range(sum(int(v.get(animal, 0)) for v in stored)))
    additions = (extra if isinstance(extra, dict) else {extra: quantity}) if extra else {}
    for animal, count in additions.items():
        own.extend({'animal': animal, 'placed_day': day + 1} for _ in range(count))
    ours, theirs = _e157_production(own, day, end), _e157_production(enemy, day, end)
    market = _get(obs, 'market', {}) or {}
    inventory = _get(market, 'inventory', {}) or {}
    shops = list(_get(_get(obs, 'town', {}), 'unlocked_shops', []) or [])
    value = 0.
    for product in ('MILK', 'WOOL', 'EGG'):
        stock = float(inventory.get(product, 10000))
        daily = 1. + 6. * sum(2 if len(_SHOP_PRODUCTS.get(s, ())) == 1 else 1
                             for s in shops if product in _SHOP_PRODUCTS.get(s, ()))
        mean_shop = 6. * sum((2 if len(v) == 1 else 1) for v in _SHOP_PRODUCTS.values()
                             if product in v) / len(_SHOP_PRODUCTS)
        for future in range(day + 1, end):
            unseen = max(0, min(8, future // 3) - len(shops))
            demand = daily + _E157_CONFIG['future_weight'] * unseen * mean_shop
            q, rival = ours[product][future], theirs[product][future]
            # A public-production scenario, not a prediction of exact sell times.
            quote = _e155_price(product, stock + .5 * (q + rival) - .5 * demand, market)
            value += q * quote
            stock -= demand
            if quote > 1:
                stock += q + rival
    if additions:
        feed = _e155_price('WHEAT', inventory.get('WHEAT', 10000), market)
        # Existing labour route is preserved; service opportunity cost is charged.
        value -= sum(count * (_E157_ANIMALS[animal][3] + (end - day - 1) * (feed + 8.))
                     for animal, count in additions.items())
    return value

def _e157_commitments(obs, action):
    farm = _farm(obs, _seat(obs))
    prices = _get(_get(obs, 'market', {}), 'prices', {}) or {}
    hires = int(_get(farm, 'hires_today', 0))
    total = 0.
    for order in action['market']:
        op = order[0]
        if op == 'HIRE':
            a, b = 1, 1
            for _ in range(hires): a, b = b, a + b
            total += a
            hires += 1
        elif op == 'BUY_LAND':
            total += 1000 * 2 ** max(0, len(_get(farm, 'unlocked_quadrants', ['NW'])) - 1)
        elif len(order) >= 3 and op.startswith('BUY_'):
            item, qty = order[1], max(0, int(order[2]))
            cost = (_E157_ANIMALS.get(item, ('', 0, 0, 300))[3] if op == 'BUY_ANIMAL'
                    else {'WHEAT': 10, 'CARROT': 20, 'TOMATO': 50,
                          'STRAWBERRY': 100, 'MELON': 80}.get(item, 100) if op == 'BUY_SEED'
                    else prices.get(item, 100) * 1.1)
            total += cost * qty
    return total

def _e157_allocate(obs, action, step):
    state = _e157_state(obs, step)
    farm = _farm(obs, _seat(obs))
    private = _get(obs, 'private', {}) or {}
    shed = dict(private.get('shed', {}) or {})
    invs = private.get('inventories', []) or []
    actors = [action['farmer'], *action['hands']]
    positions = [_get(farm, 'farmer', [0, 0]), *(_get(farm, 'hands', []) or [])]
    # Resolve purchases made on earlier turns BEFORE today's market decisions.
    for index, (order, pos) in enumerate(zip(actors, positions)):
        if order[0] == 'BUILD_PASTURE':
            goose = any(key[0] == index and value[0] == 'GOOSE' and value[1] > 0
                        for key, value in state['carrying'].items())
            if goose and _tile_at(farm, pos) is None:
                order[:] = ['BUILD_COOP']
            continue
        if len(order) < 2 or order[1] not in _E157_ANIMALS:
            continue
        original = order[1]
        if order[0] == 'PICKUP':
            qty = int(order[2]) if len(order) > 2 else 1
            queue = state['queues'][original]
            if qty <= 0 or len(queue) < qty or len(set(queue[:qty])) != 1:
                continue
            actual = queue[0]
            if pos[0] not in (4, 5) or pos[1] not in (4, 5) or shed.get(actual, 0) < qty:
                continue
            order[1] = actual
            shed[actual] -= qty
            del queue[:qty]
            state['carrying'][(index, original)] = [actual, qty]
            _E157_TELEMETRY['converted_pickups'] += qty * (actual != original)
        elif order[0] == 'PLACE':
            carry = state['carrying'].get((index, original))
            tile = _tile_at(farm, pos)
            inv = invs[index] if index < len(invs) else {}
            if (carry and carry[1] > 0 and inv.get(carry[0], 0) > 0
                    and isinstance(tile, dict)
                    and tile.get('kind') == ('COOP' if carry[0] == 'GOOSE' else 'PASTURE')
                    and not tile.get('animal')):
                order[1] = carry[0]
                carry[1] -= 1
                _E157_TELEMETRY['converted_placements'] += int(carry[0] != original)
    action['farmer'], action['hands'] = actors[0], actors[1:]
    for due, order in list(state['deferred']):
        if due <= step and len(action['market']) < 10:
            action['market'].append(list(order))
            state['deferred'].remove((due, order))
    extra_orders = []
    deferred_indices = set()
    for order_index, order in enumerate(action['market']):
        if len(order) < 3 or order[0] != 'BUY_ANIMAL' or order[1] not in _E157_ANIMALS:
            continue
        original, qty = order[1], max(0, int(order[2]))
        if step < 72 or qty == 0:
            continue
        default = ('SHEEP' if original == 'COW' and
                   list(_get(_get(obs, 'town', {}), 'unlocked_shops', []) or [])[:1] == ['YARN_STORE']
                   else original)
        options = ('COW', 'SHEEP', 'GOOSE') if _E157_CONFIG['geese'] else ('COW', 'SHEEP')
        if _E157_CONFIG.get('coherent'):
            # Keep the baseline wool allocation. Additional milk is not a
            # substitute for a small diversified wool block. Fresh pastures
            # are built AFTER the step-150 purchase; other slots reuse
            # pastures built long before the investment decision and cannot
            # lawfully become coops without changing the whole work route.
            shops = list(_get(_get(obs, 'town', {}), 'unlocked_shops', []) or [])
            options = [original]
            if original == 'COW' and 'YARN_STORE' in shops:
                options.append('SHEEP')
            if (original == 'COW' and _E157_CONFIG['geese'] and step == 150
                    and any('EGG' in _SHOP_PRODUCTS.get(s, ()) for s in shops)):
                options.append('GOOSE')
        values = {animal: _e157_projection(obs, step, animal, qty) for animal in options}
        goose_count = min(qty, int(_E157_CONFIG.get('goose_limit', 2)))
        if 'GOOSE' in options and goose_count < qty:
            # Price the actual mixed portfolio, not a two-goose forecast
            # followed by buying only one. Pickup slots are one unit each.
            values['GOOSE'] = _e157_projection(obs, step, {'GOOSE': goose_count, original: qty - goose_count})
        chosen = max(values, key=values.get)
        if values[chosen] < values[default] + _E157_CONFIG['hysteresis'] * qty:
            chosen = default
        extra_cost = max(0, _E157_ANIMALS[chosen][3] - _E157_ANIMALS[original][3]) * qty
        if extra_cost and _get(farm, 'money', 0) < _e157_commitments(obs, action) + extra_cost + 400:
            chosen = original
        order[1] = chosen
        chosen_count = goose_count if chosen == 'GOOSE' else qty
        if chosen_count < qty:
            if (len(action['market']) + len(extra_orders) - len(deferred_indices) >= 10
                    and step == 150 and _E157_CONFIG.get('coherent')):
                # The frozen complete route has ten orders at 150. Its one
                # strawberry seed is not planted until 155; turn 151 has two
                # orders and NO planting. Move that purchase by exactly one
                # turn, keeping both animal pickups at 151 and all routes.
                index = next((i for i, o in enumerate(action['market'])
                              if o == ['BUY_SEED', 'STRAWBERRY', 1]
                              and i not in deferred_indices), None)
                if index is not None:
                    deferred_indices.add(index)
                    state['deferred'].append((step + 1, list(action['market'][index])))
                    _E157_TELEMETRY['deferred_seed_orders'] += 1
            if len(action['market']) + len(extra_orders) - len(deferred_indices) >= 10:
                chosen, chosen_count, order[1] = original, qty, original
            else:
                order[2] = chosen_count
                extra_orders.append((order_index + 1, ['BUY_ANIMAL', original, qty - chosen_count]))
        state['queues'][original].extend([chosen] * chosen_count + [original] * (qty - chosen_count))
        state['changed'] |= chosen != original
        if chosen != default and _E157_TELEMETRY['first_divergence'] < 0:
            _E157_TELEMETRY['first_divergence'] = step
        state['decisions'].append({'step': step, 'original': original, 'chosen': chosen,
                                  'qty': qty, 'chosen_count': chosen_count, 'values': values})
        _E157_TELEMETRY['allocation_decisions'] += 1
        _E157_TELEMETRY['converted_units'] += chosen_count * (chosen != original)
    expanded = []
    for index, order in enumerate(action['market']):
        if index not in deferred_indices:
            expanded.append(order)
        expanded.extend(extra for after, extra in extra_orders if after == index + 1)
    action['market'] = expanded
    _E157_TELEMETRY['pending_pickups'] = sum(len(v) for v in state['queues'].values())
    _E157_TELEMETRY['pending_placements'] = sum(v[1] for v in state['carrying'].values())
    _E157_TELEMETRY['pending_market_orders'] = len(state['deferred'])
    return action

def _e157_base(obs, configuration=None):
    step = _e154_step(obs, configuration)
    if _E157_CONFIG['allocate']:
        shops = list(_get(_get(obs, 'town', {}), 'unlocked_shops', []) or [])
        if _E157_CONFIG.get('coherent') and shops[:1] == ['YARN_STORE']:
            # Preserve the complete proven first-Yarn conversion and its
            # production/sale ledger, rather than approximating it again.
            return _E157_PARENT_BASE(obs, configuration)
        action = _align_hands(_BASE_NS['agent'](obs), obs)
        fallback = _copy_action(action)
        try:
            return _e157_allocate(obs, action, step)
        except Exception:
            _E157_TELEMETRY['exceptions'] += 1
            return fallback
    return _E157_PARENT_BASE(obs, configuration)

_E156_BASE = _e157_base

def _e157_inventory(obs, action, step, configuration=None):
    shed, carried = _e155_post_field(obs, action, configuration)
    market = _get(obs, 'market', {}) or {}
    inv = _get(market, 'inventory', {}) or {}
    quantities = {}
    for item in ('MILK', 'WOOL', 'EGG'):
        stock = max(0, int(shed.get(item, 0)))
        q = 0
        while q < stock and _e155_price(item, inv.get(item, 10000) + q, market) > 1:
            q += 1
        if sum(shed.values()) + carried > 78 or _get(_farm(obs, _seat(obs)), 'money', 0) < 1800:
            q = stock
        quantities[item] = q
    remaining = dict(quantities)
    for order in action['market']:
        if len(order) > 2 and order[0] == 'SELL' and order[1] in quantities:
            order[2] = remaining.pop(order[1], 0)
    for item, qty in remaining.items():
        if qty > 0 and len(action['market']) < 10:
            action['market'].append(['SELL', item, qty])
    _E157_TELEMETRY['inventory_turns'] += 1
    return action

def e157_kaggle_entrypoint(obs, configuration=None):
    step = _e154_step(obs, configuration)
    normalized = dict(obs)
    normalized['step'] = step
    state = _e157_state(normalized, step)
    action = _E157_PARENT_ENTRY(normalized, configuration)
    fallback = _copy_action(action)
    try:
        if step >= 72 and _E157_CONFIG['service']:
            action = _e155_idle_economy(normalized, action, step)
        if step >= 192 and (_E157_CONFIG['inventory'] or
                            (_E157_CONFIG['allocate'] and state['changed'])):
            action = _e157_inventory(normalized, action, step, configuration)
            action = _e156_priority(normalized, action)
        if step >= int(_get(configuration or {}, 'episodeSteps', 720)) - 2 and _E157_CONFIG['terminal']:
            shed, _ = _e155_post_field(normalized, action, configuration)
            action['market'] = [['SELL', item, int(qty)] for item, qty in shed.items()
                                if item in _E155_PARAMS and qty > 0]
            action = _e156_priority(normalized, action)
            _E157_TELEMETRY['terminal_sales'] += sum(o[2] for o in action['market'])
        if action != fallback and _E157_TELEMETRY['first_divergence'] < 0:
            _E157_TELEMETRY['first_divergence'] = step
        return _align_hands(action, normalized)
    except Exception:
        _E157_TELEMETRY['exceptions'] += 1
        return fallback

# The official loader selects the final defined callable above.
_E155_TELEMETRY = _E157_TELEMETRY
agent = e157_kaggle_entrypoint
e157_kaggle_entrypoint.telemetry = _E157_TELEMETRY



''')


### Joint portfolio guard


In [10]:
SOURCE_PARTS.append(r'''# E158: joint portfolio selection over the two proven coherent slots.
_E158_PORTFOLIO_CONFIG = {'robust': False, 'second_goose_hurdle': 2000.0}
_E158_PARENT_AGENT = agent
_E158_PARENT_ALLOCATE = _e157_allocate
_E158_PORTFOLIO_TELEMETRY = _E157_TELEMETRY
_E158_PORTFOLIO_TELEMETRY.update({"portfolio_calls": 0, "portfolio_0": 0,
                                  "portfolio_1": 0, "portfolio_2": 0,
                                  "portfolio_scores": []})


def _e158_portfolio_score(obs, step, geese, future_weights):
    original_weight = float(_E157_CONFIG.get("future_weight", .35))
    values = []
    try:
        for weight in future_weights:
            _E157_CONFIG["future_weight"] = float(weight)
            additions = {"COW": 2 - geese}
            if geese:
                additions["GOOSE"] = geese
            values.append(_e157_projection(obs, step, additions))
    finally:
        _E157_CONFIG["future_weight"] = original_weight
    return min(values) if _E158_PORTFOLIO_CONFIG["robust"] else values[0]


def _e157_allocate(obs, action, step):
    # Reset the parent limit before every episode; choose only at the complete
    # two-unit slot.  The parent still owns all dependency and affordability checks.
    if step == 0:
        _E157_CONFIG["goose_limit"] = 2
    if step == 150:
        shops = list(_get(_get(obs, "town", {}) or {}, "unlocked_shops", []) or [])
        if any("EGG" in _SHOP_PRODUCTS.get(shop, ()) for shop in shops):
            weights = (0., .35, .70) if _E158_PORTFOLIO_CONFIG["robust"] else (.35,)
            scores = {count: _e158_portfolio_score(obs, step, count, weights)
                      for count in (0, 1, 2)}
            chosen = max(scores, key=scores.get)
            # Preserve E157's separately tested economic hurdle relative to cows.
            if scores[chosen] < scores[0] + float(_E157_CONFIG.get("hysteresis", 300.)) * 2:
                chosen = 0
            second_hurdle = float(_E158_PORTFOLIO_CONFIG.get("second_goose_hurdle", 0.))
            if chosen == 2 and scores[2] < scores[1] + second_hurdle:
                chosen = 1 if scores[1] >= scores[0] + float(
                    _E157_CONFIG.get("hysteresis", 300.)
                ) * 2 else 0
            _E157_CONFIG["goose_limit"] = chosen
            _E158_PORTFOLIO_TELEMETRY["portfolio_calls"] += 1
            _E158_PORTFOLIO_TELEMETRY["portfolio_" + str(chosen)] += 1
            if len(_E158_PORTFOLIO_TELEMETRY["portfolio_scores"]) < 8:
                _E158_PORTFOLIO_TELEMETRY["portfolio_scores"].append(
                    {"step": step, "chosen": chosen, "scores": scores,
                     "shops": shops, "robust": _E158_PORTFOLIO_CONFIG["robust"]}
                )
    return _E158_PARENT_ALLOCATE(obs, action, step)


def e158_portfolio_entrypoint(obs, configuration=None):
    return _E158_PARENT_AGENT(obs, configuration)


agent = e158_portfolio_entrypoint
agent.telemetry = _E158_PORTFOLIO_TELEMETRY
e158_portfolio_entrypoint.telemetry = _E158_PORTFOLIO_TELEMETRY
''')


In [11]:
import hashlib
import math
from pathlib import Path

HISTORICAL_SHA256 = 'fb057db8841c467d9d3ec7409de767049e906aeb9d5f0b6c77a6608e23362bd9'
for value in (TOTAL_BUFFER, SECOND_GOOSE_BUFFER):
    if not math.isfinite(value) or value < 0:
        raise ValueError("Buffers must be finite and nonnegative")

source = "".join(SOURCE_PARTS)
# Python normalizes source-string newlines. Restore the frozen file's mixed line endings
# only when the readable source is otherwise unchanged; user edits remain editable.
if hashlib.sha256(source.encode("utf-8")).hexdigest() == 'b58c152f830eef3abb75ff3006d0f3309f271733df0691ad245742029ce8636c':
    crlf_lines = set([106, 107, 108, 109, 110, 111, 114, 115, 116, 117, 118, 119, 122, 125, 126, 129, 130, 131, 132, 133, 134, 135, 138, 139, 140, 141, 142, 145, 146, 147, 148, 149, 164, 165, 166, 167, 168, 169, 170, 171, 172])
    source = "".join(line.replace("\n", "\r\n") if i in crlf_lines else line
                     for i, line in enumerate(source.splitlines(keepends=True)))
historical_source = hashlib.sha256(source.encode("utf-8")).hexdigest() == HISTORICAL_SHA256
replacements = {
    "'hysteresis': 300.0": "'hysteresis': " + repr(float(TOTAL_BUFFER) / 2),
    "'second_goose_hurdle': 2000.0": "'second_goose_hurdle': " + repr(float(SECOND_GOOSE_BUFFER)),
}
for old, new in replacements.items():
    if old != new:
        if source.count(old) != 1:
            raise ValueError("Configuration marker changed; update the build cell too: " + old)
        source = source.replace(old, new, 1)
agent_bytes = source.encode("utf-8")
compile(agent_bytes, "main.py", "exec")
AGENT_SHA256 = hashlib.sha256(agent_bytes).hexdigest()
if historical_source and (TOTAL_BUFFER, SECOND_GOOSE_BUFFER) == (600.0, 2000.0):
    assert AGENT_SHA256 == HISTORICAL_SHA256
MAIN_PATH = Path("main.py")
MAIN_PATH.write_bytes(agent_bytes)
print("Agent:", MAIN_PATH.name, "|", len(agent_bytes), "bytes")
print("SHA-256:", AGENT_SHA256)
print("Byte-exact historical S38:", AGENT_SHA256 == HISTORICAL_SHA256)


## 5. Check the complete agent

These four games run the generated file against the built-in starter, using two seeds
and both player positions. Each game loads a fresh module so state cannot leak between
runs. The check fails on exceptions, malformed actions or incomplete games.

This is a compatibility check, not a competitive benchmark. Before relying on a change,
use more seeds and stronger reacting opponents. The printed results belong to this run,
not to the historical leaderboard submission.


In [12]:
# Run tests in an isolated process; show results rather than unrelated engine startup logs.
SMOKE_PROGRAM = r'''import contextlib
import importlib.metadata
import io
import json
import time
from pathlib import Path
import sys

SMOKE_SEEDS = json.loads(sys.stdin.read())
MAIN_PATH = Path("main.py")

# Keep unrelated environment-import chatter out of the reader's output.
import_log = io.StringIO()
try:
    with contextlib.redirect_stdout(import_log), contextlib.redirect_stderr(import_log):
        from kaggle_environments import make
        from kaggle_environments.agent import get_last_callable
except Exception:
    print(import_log.getvalue()[-2000:])
    raise

ENGINE_VERSION = importlib.metadata.version("kaggle-environments")
assert ENGINE_VERSION == "1.32.7", "Unexpected game version"
print("kaggle-environments:", ENGINE_VERSION)
smoke_results = []
for seed in SMOKE_SEEDS:
    for seat in (0, 1):
        namespace = {"__name__": "s38_smoke", "__file__": str(MAIN_PATH)}
        exec(compile(MAIN_PATH.read_bytes(), "main.py", "exec"), namespace)
        entrypoint = namespace["e158_portfolio_entrypoint"]
        assert namespace["agent"] is entrypoint
        loader_entrypoint = get_last_callable(MAIN_PATH.read_text(encoding="utf-8"))
        assert loader_entrypoint.__name__ == "e158_portfolio_entrypoint"
        calls = [0]
        callback_ms = []
        def checked_agent(observation, configuration):
            start = time.perf_counter()
            action = entrypoint(observation, configuration)
            assert action == loader_entrypoint(observation, configuration), "File-loader action mismatch"
            callback_ms.append((time.perf_counter() - start) * 1000)
            calls[0] += 1
            assert isinstance(action, dict) and set(action) == {"farmer", "hands", "market"}
            assert len(action["market"]) <= 10
            assert len(action["hands"]) == len(observation["farms"][seat]["hands"])
            assert all(isinstance(order, list) and len(order) > 0
                       for order in [action["farmer"], *action["hands"], *action["market"]])
            assert namespace["_E157_TELEMETRY"]["exceptions"] == 0
            return action
        env = make("kaggriculture", configuration={"episodeSteps":720, "seed":seed}, debug=False)
        players = [checked_agent, "starter"] if seat == 0 else ["starter", checked_agent]
        env.run(players)
        statuses = [s.status for s in env.state]
        assert statuses == ["DONE", "DONE"], (seed, seat, statuses)
        assert len(env.steps) == 720 and calls[0] == 719, (len(env.steps), calls)
        row = {"seed":seed, "seat":seat, "status":statuses[seat],
               "bank":env.state[seat].reward, "opponent_bank":env.state[1-seat].reward,
               "callbacks":calls[0], "max_callback_ms":round(max(callback_ms), 3)}
        smoke_results.append(row)
        print("seed={seed} seat={seat} {status} | bank={bank:.0f} vs {opponent_bank:.0f} | "
              "{callbacks} calls, max {max_callback_ms:.3f} ms".format(**row))
del env, namespace, entrypoint, loader_entrypoint, checked_agent
print("S38_SMOKE_RESULT:" + json.dumps({"engine":ENGINE_VERSION,"games":smoke_results}))
'''

import io
import json
import subprocess
import sys

result = subprocess.run([sys.executable, "-c", SMOKE_PROGRAM],
                        input=json.dumps(SMOKE_SEEDS), capture_output=True,
                        text=True, timeout=240)
if result.returncode:
    print((result.stdout + result.stderr)[-3000:])
    raise RuntimeError("Full-agent check failed; no submission archive was built")
records = [line.split(":",1)[1] for line in result.stdout.splitlines()
           if line.startswith("S38_SMOKE_RESULT:")]
assert len(records) == 1, "Missing complete test results"
test_result = json.loads(records[0])
ENGINE_VERSION = test_result["engine"]
smoke_results = test_result["games"]
print("kaggle-environments:", ENGINE_VERSION)
for row in smoke_results:
    print("seed={seed} seat={seat} {status} | bank={bank:.0f} vs {opponent_bank:.0f} | "
          "{callbacks} calls, max {max_callback_ms:.3f} ms".format(**row))
print("File-loader actions matched the intended entrypoint on every call.")


## 6. Build the submission archive

`submission.tar.gz` contains one root file, `main.py`. The check below reads it back and
verifies every byte. The source and license/credit files are also available separately
in the notebook's outputs. The agent itself needs only the Python standard library.

After a successful saved run, select `submission.tar.gz` when submitting your notebook
version. Changing settings or source creates a new policy; the historical 2615.0 does
not become evidence for that modified policy.


In [13]:
import gzip
import tarfile

archive_buffer = io.BytesIO()
with gzip.GzipFile(fileobj=archive_buffer, mode="wb", filename="", mtime=0) as gz:
    with tarfile.open(fileobj=gz, mode="w", format=tarfile.USTAR_FORMAT) as archive:
        member = tarfile.TarInfo("main.py")
        member.size = len(agent_bytes)
        member.mode = 0o644
        member.mtime = 0
        archive.addfile(member, io.BytesIO(agent_bytes))
ARCHIVE_PATH = Path("submission.tar.gz")
ARCHIVE_PATH.write_bytes(archive_buffer.getvalue())
with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    member = archive.getmember("main.py")
    assert member.isfile() and not member.issym()
    assert archive.extractfile(member).read() == MAIN_PATH.read_bytes() == agent_bytes
Path("LICENSE.txt").write_text('\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', encoding="utf-8")
Path("NOTICE.txt").write_text('S38 Goose Portfolio\nDerived from prvsiyan, Kaggriculture Frontier | The Moon Counts Melons,\nKaggle notebook V29 / scriptVersionId 345469411 (internal policy V189C).\nhttps://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-the-moon-counts-melons?scriptVersionId=345469411\nUpstream code SHA-256: 4158d6c803250c3e496c1dbc9bfea1c915c74cb4e4cc1554de99d1fce9e58352\nUpstream license: Apache-2.0. The inherited module credits a Ramesh public-mode action schedule.\nModifications by Dmitrii Gluzdov: market support, livestock valuation,\ncompatible purchase/pickup/placement allocation, and the joint portfolio guard.\nHistorical S38 code is preserved unchanged at the default settings.\nNotebook explanation and build/test cells added with AI assistance in September 2026.\nUpstream authors retain their copyrights. See LICENSE.txt.\n', encoding="utf-8")

receipt = {"agent_sha256":AGENT_SHA256, "historical_s38":AGENT_SHA256==HISTORICAL_SHA256,
           "archive_sha256":hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest(),
           "engine":ENGINE_VERSION, "smoke_results":smoke_results,
           "competition_submission_made":False}
Path("build_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n",encoding="utf-8")
print("Ready:", ARCHIVE_PATH.name, "| verified root main.py")
print("Archive SHA-256:", receipt["archive_sha256"])


## 7. What the historical results tell us

| Evidence | Result | What it measures |
|---|---|---|
| S38, submission `55859195` | Historical public LB **2615.0** | One leaderboard run |
| Same code, submission `55918268` | **1833.3** in the same account check | A different leaderboard run |
| Archived local opponent panel | **38 wins / 4 losses / 6 ties** | Those 48 games only |
| Archived server replay check | **719/719 actions matched in both seats** | Reproduction of one recorded game |

The archived local panel produced the same actions with and without the second-goose
buffer. It therefore does not show that the buffer improves results. Different LB
ratings are not a controlled comparison either.

A useful next test is to compare the default with `SECOND_GOOSE_BUFFER = 0`, using the
same fresh seeds and opponents in both positions. Record how often the decision changes,
whether the animals actually get placed, and final bank differences. Keep development
games separate from the games used to evaluate the final choice.

## Credits

- [The Moon Counts Melons — prvsiyan, V29](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-the-moon-counts-melons?scriptVersionId=345469411):
  the inherited controller, including its Ramesh-labelled public-mode schedule.
- [Kaggriculture: Getting Started](https://www.kaggle.com/code/bovard/kaggriculture-getting-started):
  environment, agent interface and submission workflow.

Released under Apache-2.0; `LICENSE.txt` and `NOTICE.txt` are included in the outputs.
AI assistance was used for implementation, testing and editing. The explanatory examples
are synthetic; the full-game check above runs the actual generated agent.
